# Trend Tracker — Notebook 03: Insights Generation

Runs the full analytical pipeline on the enriched corpus, from TF-IDF and NMF
topic discovery through LLM synthesis, verification, packaging, and DOCX report
generation.

**Run order:** `01_token_preprocess` → `02_semantic_enrichment` → `03_insights_generation`

| Step | What | Key outputs |
|------|------|-------------|
| 1 | Build TF-IDF matrices | in-memory for Steps 2–4 |
| 2 | Quality checkpoint 2 | `quality/quality_cp2.json` |
| 3 | Category TF-IDF | `analysis/category_tfidf.csv` |
| 4 | NMF topic discovery | `analysis/nmf_topics.csv`, `analysis/project_topic_bridge.csv` |
| 5 | LLM topic labeling | `analysis/llm_topic_labels.json` |
| 6 | Synthesis | `analysis/llm_synthesis_*.txt` |
| 7 | Structured insight extraction | `insights/insights_candidates.json` |
| 8 | Topic verification | (updates insights in-memory) |
| 9 | Evidence tables | `insights/insights_flat.csv`, `insights/insight_topic_support_candidates.csv` |
| 10 | Packaging, dedupe & topline | (updates curated insights in-memory) |
| 11 | Final outputs & manifests | `reports/trend_tracker_report.docx`, `insights/insights_structured.json` |

---
## Setup and Configuration

In [1]:
import sys
import json
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

import pandas as pd
import numpy as np

ROOT = Path(".")
sys.path.insert(0, str(ROOT))

import importlib.util as _ilu, sys as _sys
_u = next(Path(".").glob("utils*.py"))
if _u.stem != "utils":
    _s = _ilu.spec_from_file_location("utils", _u); _m = _ilu.module_from_spec(_s); _s.loader.exec_module(_m); _sys.modules["utils"] = _m
    
from utils import (
    load_cfg,
    write_json,
    artifact_meta,
    build_run_output_path,
    get_run_date,
    canonicalize_filter_spec,
    get_filter_fields_key,
    get_run_id,
    apply_filters,
    ensure_warning_file,
    append_warning,
    get_llm_client,
    start_stage_manifest,
    finalize_stage_manifest,
    build_pipeline_manifest,
    tokens_to_str,
    make_vec,
    add_bin,
    group_key,
    build_project_topic_bridge,
    quality_report,
    cat_tfidf_slice,
    nmf_one,
    build_input,
    _label_with_retry,
    _norm_group_value,
    _safe_topic_id,
    build_topic_lines,
    _call_with_retry,
    synthesize_one_group,
    strip_json_fences,
    normalize_insight,
    build_bridge_lookup,
    build_label_index,
    build_verified_insight_tables,
    apply_deterministic_packaging,
    dedupe_packaged_insights,
    assign_topline_sections_simple,
    build_structured_from_curated,
    build_looker_project_url,
    build_packaged_report_docx,
    project_insight_for_saved_candidates,
    _verify_insight_list,
    resolve_params_path,
)

CFG_PATH = resolve_params_path()
CFG      = load_cfg(CFG_PATH)

try:
    MANIFEST_CFG_PATH = str(CFG_PATH.relative_to(ROOT))
except ValueError:
    MANIFEST_CFG_PATH = str(CFG_PATH)
ct       = CFG["tfidf"]
client   = get_llm_client()

# Load the enriched corpus produced by NB02. To use the unenriched baseline
# instead, change the path to 05_filtered.parquet.
raw_df = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")

# Stopwords for the quality gate at Step 2. Loaded from params.yaml so the
# list can be extended without touching notebook code.
STOPWORDS = CFG["quality"]["stopword_violation_list"]

print(f"Loaded {len(raw_df):,} projects  |  {raw_df.shape[1]} columns")

Loaded 2,244,916 projects  |  45 columns


---
## Parameters

All runtime configuration is resolved here from `params.yaml`. Edit `params.yaml`
to change behaviour; do not hardcode values in downstream cells.

In [2]:
# ── Analysis scope ────────────────────────────────────────────────────────────
GROUPBY_FIELD  = CFG["analysis"]["group_by"]
FILTER_LOGIC   = CFG["analysis"].get("filter_logic", "and")
FILTERS        = CFG["analysis"].get("filters", [])
EXCLUDE_GROUPS = CFG["analysis"]["exclude_groups"]
REVIEW_GROUP   = CFG["analysis"]["review_group"]

# Apply declarative filters first so all corpus-scope checks run on the
# right population.
df, filter_summary = apply_filters(raw_df, FILTER_LOGIC, FILTERS)
if filter_summary["no_rows_after_filter"]:
    raise ValueError("No rows remain after applying analysis filters — check params.yaml.")

# ── v1 feature guards ─────────────────────────────────────────────────────────
# Binned/time-slice mode: topic identity and downstream traceability (bridge
# table, label index, source_topics) are keyed by group + topic_id only.
# Adding a bin dimension would cause silent topic-id collisions across bins
# within the same group. Guard here until topic identity is bin-aware.
if CFG["analysis"].get("bins", []):
    raise ValueError(
        "analysis.bins is non-empty: binned/time-slice mode is not supported "
        "in v1. Topic identity and source_topic traceability are not bin-aware. "
        "Set analysis.bins: [] to run without time bucketing."
    )

# review_group: restricting to a single group would bypass cross-group
# synthesis and require conditional output handling that is not wired in v1.
# Disable until the cross-group path is made optional.
if REVIEW_GROUP is not None:
    raise ValueError(
        f"analysis.review_group is set to {REVIEW_GROUP!r}: single-group review "
        "mode is not supported in v1. Set analysis.review_group: null to run "
        "against all groups."
    )

# ── Group scope: apply exclude_groups once, here, for the whole pipeline ──────
# Removing excluded groups from df means every downstream step (TF-IDF, NMF,
# labeling, synthesis) automatically excludes them. No per-step filtering needed.
if EXCLUDE_GROUPS:
    n_before = len(df)
    df = df[
        ~df[GROUPBY_FIELD].astype(str).isin([str(g) for g in EXCLUDE_GROUPS])
    ].reset_index(drop=True)
    if df.empty:
        raise ValueError(
            f"No rows remain after excluding groups {EXCLUDE_GROUPS} — "
            "check analysis.exclude_groups in params.yaml."
        )
    print(f"exclude_groups: {n_before - len(df):,} rows removed "
          f"({len(EXCLUDE_GROUPS)} group(s): {EXCLUDE_GROUPS})")

# ── Run identity ───────────────────────────────────────────────────────────────
# RUN_DATE and RUN_ID are assigned BEFORE OUT() is defined because OUT() closes
# over them. Reversing this order causes a NameError at the OUT() call site.
FILTER_SPEC       = canonicalize_filter_spec(FILTER_LOGIC, FILTERS)
FILTER_FIELDS_KEY = get_filter_fields_key(FILTERS)
RUN_DATE          = get_run_date()
RUN_ID            = get_run_id(GROUPBY_FIELD, FILTER_SPEC)

def OUT(subdir, fname):
    return build_run_output_path(
        subdir=subdir, fname=fname,
        groupby_field=GROUPBY_FIELD, run_date=RUN_DATE, run_id=RUN_ID,
    )

# ── Analysis description and topic/slice settings ─────────────────────────────
GROUP_DESCRIPTION = CFG["analysis"]["group_descriptions"].get(
    GROUPBY_FIELD,
    f"groups defined by '{GROUPBY_FIELD}' in DonorsChoose data",
)
MIN_SHARED    = CFG["analysis"]["nmf_min_shared"]
MIN_COVERAGE  = CFG["analysis"]["min_coverage"]
BASE_N_COMPONENTS = CFG["nmf"]["n_components"]
CAT_TFIDF_TOP_N   = CFG["analysis"]["cat_tfidf_top_n"]

SLICE_RULES   = CFG["analysis"]["slice_rules"]
VERIFY_CFG    = CFG["analysis"]["verification"]
PACKAGING_CFG = CFG["analysis"]["packaging"]
DEDUPE_CFG    = CFG["analysis"]["dedupe"]

# group_project_counts is computed on the scoped df (post-filter, post-exclude)
# so small_slice_mode reflects the actual groups that will run.
group_project_counts = (
    df[[GROUPBY_FIELD, "project_id"]]
    .dropna(subset=["project_id"])
    .drop_duplicates()
    .groupby(GROUPBY_FIELD)["project_id"]
    .nunique()
)
median_group_projects = float(group_project_counts.median()) if not group_project_counts.empty else 0.0
SMALL_SLICE_MODE = median_group_projects < SLICE_RULES["small_slice_cutoff"]
SLICE_RULES      = {**SLICE_RULES, "small_slice_mode": SMALL_SLICE_MODE}

MIN_GROUP_PROJECTS_EFFECTIVE = max(
    CFG["analysis"]["min_group_projects"],
    SLICE_RULES["min_group_projects"],
)
CROSS_MIN_INSIGHTS     = SLICE_RULES["cross_min_insights"]
CROSS_MAX_INSIGHTS     = SLICE_RULES["cross_max_insights"]
PER_GROUP_MIN_INSIGHTS = SLICE_RULES["per_group_min_insights"]
PER_GROUP_MAX_INSIGHTS = SLICE_RULES["per_group_max_insights"]

# ── LLM settings ──────────────────────────────────────────────────────────────
N_REPRESENTATIVE          = CFG["llm"]["n_representative_snippets"]
TOP_TERMS_IN_PROMPT       = CFG["llm"]["top_terms_in_prompt"]
SYNTHESIS_TOP_TERMS_COUNT = CFG["llm"]["synthesis_top_terms_count"]
# MAX_RETRIES is passed to all LLM call sites so retry behaviour is tunable
# from params.yaml without touching notebook code.
MAX_RETRIES = CFG["llm"]["max_retries"]

MODEL_LABELING  = CFG["models"]["labeling"]
MODEL_SYNTHESIS = CFG["models"]["synthesis"]
MODEL_VERIFY    = CFG["models"]["verify"]

MAX_WORKERS          = CFG["analysis"]["synthesis_max_workers"]
LABELING_MAX_WORKERS = CFG["analysis"]["labeling_max_workers"]

# ── Output settings ────────────────────────────────────────────────────────────
LOOKER_BASE_URL     = CFG["output"]["looker_base_url"]
LOOKER_FILTER_FIELD = CFG["output"]["looker_filter_field"]
LOOKER_FIELDS       = CFG["output"]["looker_fields"]
LOOKER_LIMIT        = int(CFG["output"].get("looker_limit", 500))
LOOKER_ID_LIMIT     = int(CFG["output"].get("looker_id_limit", 100))

# CSV_MAX_IDS_PER_INSIGHT: ranked project IDs stored per insight row in
# insights_flat.csv. Passed to build_verified_insight_tables().
CSV_MAX_IDS_PER_INSIGHT = int(CFG["output"]["csv_max_ids_per_insight"])

# MAIN_MIN_VERIFICATION_RATIO: eligibility gate for main-section topline
# placement. Passed to assign_topline_sections_simple().
MAIN_MIN_VERIFICATION_RATIO = PACKAGING_CFG["main_min_verification_ratio"]

# bins are guarded out above; group_cols is always [GROUPBY_FIELD] in v1.
group_cols = [GROUPBY_FIELD]

# ── Warnings, filter provenance, stage manifest ────────────────────────────────
WARNINGS_PATH       = OUT("metadata", "warnings_03.jsonl")
FILTER_SPEC_PATH    = OUT("metadata", "filter_spec.json")
FILTER_SUMMARY_PATH = OUT("metadata", "filter_summary.json")
COPIED_CONFIG_PATH  = OUT("metadata", CFG_PATH.name)

ensure_warning_file(WARNINGS_PATH)
write_json(FILTER_SPEC_PATH, {
    "schema_version": "v1", "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD, "filter_fields_key": FILTER_FIELDS_KEY,
    "filter_logic": FILTER_LOGIC, "filters": FILTERS,
})
write_json(FILTER_SUMMARY_PATH, {
    "schema_version": "v1", "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD, **filter_summary,
})
shutil.copy2(CFG_PATH, COPIED_CONFIG_PATH)

STAGE_MANIFEST = start_stage_manifest(
    stage_name="03_insights_generation",
    notebook_file="03_insights_generation_v1.ipynb",
    config_path=MANIFEST_CFG_PATH,
    
    run_id=RUN_ID,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
)

print(f"RUN_ID            = {RUN_ID}")
print(f"GROUPBY_FIELD     = {GROUPBY_FIELD!r}")
print(f"FILTER_FIELDS_KEY = {FILTER_FIELDS_KEY}")
print(f"Filtered rows     = {len(df):,} / {len(raw_df):,}")
print(f"small_slice_mode  = {SMALL_SLICE_MODE}  (median group = {median_group_projects:.1f})")
print(f"Output root       = OUTPUTS/runs/{GROUPBY_FIELD}/{RUN_DATE}/{RUN_ID}/")

RUN_ID            = 20260410_111800_project_category_77e319db
GROUPBY_FIELD     = 'project_category'
FILTER_FIELDS_KEY = funded_date
Filtered rows     = 420,625 / 2,244,916
small_slice_mode  = False  (median group = 22972.0)
Output root       = OUTPUTS/runs/project_category/2026-04-10/20260410_111800_project_category_77e319db/


---
## Step 1 — Build TF-IDF Matrices

Fits one TF-IDF vectorizer per n-gram range on the full filtered corpus and
keeps the resulting sparse matrices in memory. All downstream steps reuse these
objects rather than refitting.

Trigrams are built when `ngrams.max_n >= 3` in `params.yaml`.

In [3]:
docs = df["tokens"].apply(tokens_to_str).tolist()

# Build one matrix per n-gram range. X_unigram_bigram is the primary matrix
# for category TF-IDF and NMF; the others are retained for debugging and
# future analytical passes.
specs = {
    "X_unigram":        (1, 1),
    "X_bigram":         (2, 2),
    "X_unigram_bigram": (1, 2),
}
if CFG["ngrams"]["max_n"] >= 3:
    specs["X_trigram"] = (3, 3)

matrices, vecs = {}, {}
for name, rng in specs.items():
    vec = make_vec(ct["min_df"], ct["max_df"], rng)
    matrices[name] = vec.fit_transform(docs)
    vecs[name] = vec
    sz, nnz = matrices[name].shape, matrices[name].nnz
    print(f"  {name:20s}: shape={sz}  sparsity={1 - nnz / (sz[0] * sz[1]):.3f}")

# Spot-check first features to catch bad filtering or token drift.
for name, vec in vecs.items():
    print(f"  {name:20s} first feats → {vec.get_feature_names_out()[:10].tolist()}")

  X_unigram           : shape=(420625, 7083)  sparsity=0.993
  X_bigram            : shape=(420625, 360566)  sparsity=1.000
  X_unigram_bigram    : shape=(420625, 367649)  sparsity=1.000
  X_trigram           : shape=(420625, 152428)  sparsity=1.000
  X_unigram            first feats → ['aac', 'aba', 'abandon', 'abc', 'abcs', 'ability', 'able', 'abound', 'above', 'abraham']
  X_bigram             first feats → ['aac app', 'aac augmentative', 'aac communicate', 'aac communication', 'aac device', 'aac system', 'aac tool', 'aba practice', 'abc book', 'abc center']
  X_unigram_bigram     first feats → ['aac', 'aac app', 'aac augmentative', 'aac communicate', 'aac communication', 'aac device', 'aac system', 'aac tool', 'aba', 'aba practice']
  X_trigram            first feats → ['aac augmentative alternative', 'aac communicate world', 'aac device communicate', 'aac device gesture', 'aba practice staff', 'ability absorb information', 'ability absorb retain', 'ability access basic', 'ability 

---
## Step 2 — Quality Checkpoint 2

Gate before LLM calls. Review before proceeding:

- **No stopword violations** — if terms from `quality.stopword_violation_list`
  appear in the top-200 vocab, re-run NB01 with tighter frequency thresholds.
- **Reasonable token distribution** — `p50` should be in the 20–60 range.
- **Matrix sparsity** — very high sparsity on the bigram matrix suggests
  `tfidf.min_df` may be too restrictive.

In [4]:
# Pass the configured stopword list so the gate reflects params.yaml rather
# than the HARD_STOPWORDS fallback in utils.py.
qr2 = quality_report(
    df, label="cp2",
    matrices=matrices,
    save_path=OUT("quality", "quality_cp2.json"),
    stopwords=STOPWORDS,
)


=======================================================  [cp2]
  Projects : 420,625
  Tok/proj : min=5  p50=45  max=97
  Vocab    : 7,609 unique tokens
  X_unigram           : shape=[420625, 7083]  sparsity=0.993
  X_bigram            : shape=[420625, 360566]  sparsity=1.000
  X_unigram_bigram    : shape=[420625, 367649]  sparsity=1.000
  X_trigram           : shape=[420625, 152428]  sparsity=1.000
  Stopwords: FAIL — ['project', 'grade', 'teacher', 'education']



---
## Step 3 — Category TF-IDF

The vectorizer is fit **once** on the full corpus; category slices are scored
by index. This avoids the string-comparison bug (identical token sets across
projects would misclassify rows) and is much faster than refitting per slice.

**Contrast** = token prevalence in this category minus prevalence outside it.

Time bins: defined in `params.yaml` under `analysis.bins`; leave empty for
the full date range.


In [5]:
# ── Category TF-IDF ────────────────────────────────────────────────────────
# Score each eligible group against the rest of the corpus using a shared matrix.

top_n = CAT_TFIDF_TOP_N
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

df_work = df.copy().reset_index(drop=True)
all_docs = df_work["tokens"].apply(tokens_to_str).tolist()

vec_cat = make_vec(ct["min_df"], ct["max_df"], tuple(ct["ngram_range"]))
X_full = vec_cat.fit_transform(all_docs)
feat = vec_cat.get_feature_names_out()
idf_vals = vec_cat.idf_

rows = []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj:
        continue

    kd = group_key(keys, group_cols)
    top = cat_tfidf_slice(
        sub.index,
        df_index=df_work.index,
        X_full=X_full,
        feat=feat,
        idf_vals=idf_vals,
        top_n=top_n,
    )
    for col, val in kd.items():
        top.insert(0, col, val)
    rows.append(top)

if not rows:
    raise RuntimeError(
        f"No groups met min_proj={min_proj} threshold — lower min_group_projects in params.yaml"
    )

cat_tfidf_df = pd.concat(rows, ignore_index=True)
cat_tfidf_df.to_csv(OUT("analysis", "category_tfidf.csv"), index=False)

print(f"{len(cat_tfidf_df):,} rows  |  {cat_tfidf_df[group_cols[0]].nunique()} groups")
cat_tfidf_df.head(10)

420 rows  |  14 groups


,project_category,token,tf,idf,tfidf,prevalence,contrast,project_count
0,Art Supplies,art,0.024475,3.385000,0.082848,0.443162,0.381143,14705
1,Art Supplies,paint,0.015290,4.707079,0.071973,0.199566,0.190008,6622
2,Art Supplies,supply,0.017763,2.650745,0.047085,0.407480,0.234038,13521
3,Art Supplies,color,0.012994,3.588745,0.046631,0.223254,0.160829,7408
4,Art Supplies,creative,0.013385,3.454842,0.046245,0.247664,0.175646,8218
5,Art Supplies,creativity,0.013352,3.349645,0.044723,0.263064,0.182023,8729
6,Art Supplies,express,0.010714,3.966923,0.042501,0.175637,0.134813,5828
7,Art Supplies,craft,0.008659,4.824281,0.041774,0.113977,0.100037,3782
8,Art Supplies,marker,0.010716,3.755995,0.040248,0.175276,0.121301,5816
9,Art Supplies,paper,0.011279,3.514531,0.039640,0.199747,0.129027,6628


---
## Step 4 — NMF Topic Discovery

NMF is fit independently per group so the dominant vocabulary axis in one group
does not suppress signal in others. Topics are treated as evidence candidates
for LLM synthesis — not as stable theme definitions.

The NMF weight matrix `W` records how strongly each project loads on each topic.
Step 5 uses the top-weight projects per topic as representative snippets rather
than sampling randomly.

In [6]:
# ── Groupwise NMF topics ───────────────────────────────────────────────────
# Fit one NMF model per eligible group and keep both topic-level and project-level outputs.

cn = CFG["nmf"]
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

all_topics, all_weights = [], []
df_work = df.copy().reset_index(drop=True)

NMF_GROUPS_SKIPPED = []
NMF_GROUPS_FAILED = []

for keys, sub in df_work.groupby(group_cols, observed=True):
    kd = group_key(keys, group_cols)
    group_value = kd[GROUPBY_FIELD]

    # Skip groups that are too small to support stable topic extraction.
    if len(sub) < min_proj:
        NMF_GROUPS_SKIPPED.append(str(group_value))
        continue

    group_docs = sub["tokens"].apply(tokens_to_str).tolist()
    pids = sub["project_id"].tolist()

    try:
        topics, W, nmf_meta = nmf_one(
            group_docs,
            ct_cfg=ct,
            cn_cfg=cn,
            base_n_components=BASE_N_COMPONENTS,
            slice_rules=SLICE_RULES,
        )
    except Exception as e:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF failed for group '{group_value}'",
            context={"group": group_value, "error": str(e)},
        )
        continue

    # None return means the slice was too thin; kept separate from exceptions for QA.
    if topics is None or W is None:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF skipped group '{group_value}' because the slice was too thin",
            context={"group": group_value, **(nmf_meta or {})},
        )
        continue

    for col, val in kd.items():
        topics[col] = val
    all_topics.append(topics)

    # Persist the full W ranking so later cells can recover top projects per topic.
    for tid in range(W.shape[1]):
        order = W[:, tid].argsort()[::-1]
        for rank, idx in enumerate(order):
            all_weights.append(
                {
                    **kd,
                    "topic_id": tid,
                    "project_id": pids[idx],
                    "weight": float(W[idx, tid]),
                    "rank": rank,
                }
            )

if not all_topics:
    raise RuntimeError(
        "No groups produced NMF topics — check n_components vs retained vocab, "
        "or lower min_group_projects"
    )

topics_df = pd.concat(all_topics, ignore_index=True)
weights_df = pd.DataFrame(all_weights)

topics_df.to_csv(OUT("analysis", "nmf_topics.csv"), index=False)
weights_df.to_csv(OUT("analysis", "nmf_weights.csv"), index=False)

print(f"{len(topics_df):,} topics across {topics_df[group_cols[0]].nunique()} groups")

# Build the project-topic bridge once so downstream evidence collection can reuse it.
threshold = CFG["analysis"]["topic_assignment_threshold"]
project_topic_bridge_df = build_project_topic_bridge(
    weights_df,
    GROUPBY_FIELD,
    threshold,
)
project_topic_bridge_df.to_csv(OUT("analysis", "project_topic_bridge.csv"), index=False)
print(
    "Bridge: "
    f"{len(project_topic_bridge_df):,} project-topic assignments "
    f"(topic_share >= {threshold})"
)

topics_df.head(6)

/opt/miniconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 400 reached. Increase it to improve convergence.
  warnings.warn(


258 topics across 14 groups
Bridge: 295,748 project-topic assignments (topic_share >= 0.3)


,topic_id,top_terms,top_weights,project_category
0,0,"[science, concept, understand, hand, model, st...","[1.9522776460621767, 1.112231392046571, 1.0978...",Art Supplies
1,1,"[problem, solve, problem solve, thinke, critic...","[1.2451410866887977, 1.2235493631126695, 1.126...",Art Supplies
2,2,"[motor, fine, fine motor, motor skill, skill, ...","[1.3590204186841848, 1.316954266613724, 1.2934...",Art Supplies
3,3,"[calm, stress, focus, emotion, safe, mental, h...","[0.8845533176549906, 0.6560119639905994, 0.590...",Art Supplies
4,4,"[pencil, marker, crayon, color, color pencil, ...","[1.4087894556335114, 1.2248897096081048, 1.005...",Art Supplies
5,5,"[paint, art, artist, medium, artwork, techniqu...","[1.1952658999763233, 1.068565778224781, 0.7843...",Art Supplies


In [7]:
# ── Cross-group universal themes ───────────────────────────────────────────
# Identify term bundles that recur across groups after topic extraction.

# Select the largest group as the cross-group reference point for any
# group-specific context needed in downstream steps.
REFERENCE_GROUP = df_work.groupby(GROUPBY_FIELD).size().idxmax()

records = [(r[group_cols[0]], frozenset(r.top_terms)) for _, r in topics_df.iterrows()]

# theme_cats maps a shared term bundle to the set of groups where it recurs.
theme_cats = defaultdict(set)
for i, (ci, si) in enumerate(records):
    for cj, sj in records[i + 1:]:
        if ci == cj:
            continue
        shared = si & sj
        if len(shared) >= MIN_SHARED:
            key = frozenset(shared)
            theme_cats[key] |= {ci, cj}

rows = sorted(theme_cats.items(), key=lambda x: -len(x[1]))
seen = []
deduped = []
for terms, cats in rows:
    if len(cats) < MIN_COVERAGE:
        continue
    if not any(terms <= prior for prior in seen):
        deduped.append(
            {
                "theme": ", ".join(sorted(terms)[:5]),
                "n_groups": len(cats),
                "categories": sorted(cats),
            }
        )
        seen.append(terms)

cross_group_df = pd.DataFrame(deduped).reset_index(drop=True)
cross_group_df.to_csv(OUT("analysis", "cross_group_themes.csv"), index=False)
cross_group_df

,theme,n_groups,categories
0,"income, low, low income, title",9,"[Art Supplies, Classroom Basics, Educational K..."
1,"goal, special, special education",9,"[Books, Classroom Basics, Computers & Tablets,..."
2,"problem, problem solve, solve",8,"[Art Supplies, Books, Classroom Basics, Educat..."
3,"family, income, income family, low, low income",8,"[Art Supplies, Classroom Basics, Food, Clothin..."
4,"emotional, self, social, social emotional",8,"[Art Supplies, Books, Classroom Basics, Comput..."
5,"appreciate, consider, donate, donation, please",8,"[Books, Classroom Basics, Computers & Tablets,..."
6,"enhance, environment, essential, experience, f...",7,"[Art Supplies, Classroom Basics, Computers & T..."
7,"appreciate, consider, donate, donation, greatly",7,"[Art Supplies, Classroom Basics, Computers & T..."
8,"emotion, emotional, social, social emotional",7,"[Books, Classroom Basics, Computers & Tablets,..."
9,"english, language, learner",7,"[Books, Classroom Basics, Computers & Tablets,..."


---
## Step 5 — LLM Topic Labeling

One API call per topic using compressed input — never raw essay text at scale.
Representative snippets are selected by NMF weight (highest-loading projects),
not randomly. Parallel execution via `ThreadPoolExecutor`.

Parse failures are stored with enough metadata to debug later; failed topics are
excluded from synthesis but do not halt the run.

In [8]:
# ── LLM topic labeling ─────────────────────────────────────────────────────
# Convert raw NMF topics into human-readable labels/descriptions with retry logic.

SYSTEM = (
    "You are an NLP analyst reviewing NMF topic clusters from DonorsChoose teacher "
    "essays. Respond ONLY with a single valid JSON object. No preamble. No markdown fences.\n\n"

    "Token naming conventions you may see in topic terms:\n"
    "  __framing_[name]__ = a rhetorical framing signal injected during preprocessing\n"
    "  __cat_[name]__ = a subject matter category token\n"
    "  __sub_[name]__ = a subcategory token\n"
    "These are meaningful signals. Use their meaning when clearly supported, but do not let them "
    "override stronger concrete evidence from repeated terms and snippets.\n"
    "Never output raw tokens such as __framing_*__, __cat_*__, __sub_*__, or snake_case token names. "
    "Always translate such signals into plain English.\n\n"

    "Your job is to produce a stable canonical topic label and one dense grounded description.\n"
    "Do not optimize for cleverness, vividness, or donor-facing insight language. "
    "Optimize for specificity, repeatability, plain-English clarity, and preservation of useful evidence.\n\n"

    "Rules for proposed_label:\n"
    "- proposed_label must be a short canonical noun phrase, usually 3 to 7 words.\n"
    "- Prefer the most specific defensible mechanism, request type, intervention, "
    "classroom routine, or use case.\n"
    "- Do not try to pack every nuance into proposed_label.\n"
    "- Do not collapse topics into broad umbrella labels like technology, literacy, "
    "engagement, classroom supplies, or social-emotional learning if the evidence "
    "supports a narrower interpretation.\n"
    "- Preserve concrete signals such as named programs, pedagogies, student populations, "
    "classroom routines, and rhetorical framing when clearly supported.\n\n"

    "Rules for description:\n"
    "- description must be exactly one sentence.\n"
    "- description must be plain English and evidence-led.\n"
    "- Use this order when supported by the evidence: "
    "(1) dominant mechanism or request pattern, "
    "(2) classroom function or use case, "
    "(3) population or setting, "
    "(4) framing signal in plain English.\n"
    "- Prefer concrete natural-language phrasing over abstract wording.\n"
    "- Do not write implications, recommendations, or donor-facing interpretation.\n"
    "- Do not use raw preprocessing tokens or token-like jargon in description.\n\n"

    "Rules for notes:\n"
    "- notes should usually be empty.\n"
    "- Use notes only for real edge cases.\n"
    "- If coherence_flag is mixed, briefly name the colliding subthemes.\n"
    "- If coherence_flag is redundant, briefly state the nature of the overlap.\n"
    "- If coherence_flag is unclear, briefly state why the topic is too scattered.\n"
    "- Do not restate the description in notes.\n\n"

    "coherence_flag definitions:\n"
    "  coherent   — one dominant theme, mechanism, population, or use case clearly leads, "
    "even if secondary signals are present.\n"
    "  mixed      — no single dominant theme clearly leads and two or more distinguishable "
    "subthemes are truly colliding.\n"
    "  redundant  — this topic's terms and snippets substantially duplicate another "
    "topic in the same group, not merely overlap in subject area.\n"
    "  unclear    — terms are too scattered or generic to support a defensible label.\n\n"

    "Important tie-breaker:\n"
    "- If one theme is primary and another is secondary, mark the topic coherent, not mixed, "
    "and keep notes empty unless the secondary signal is important to preserve.\n"
    "- If two phrasings are both plausible, choose the more literal, reusable, and plain-English one."
)

PROMPT = (
    f"Group ({GROUPBY_FIELD}): {{group_value}}{{bin_line}}\n"
    "Topic {topic_id}\n"
    "Top unigrams : {unigrams}\n"
    "Top bigrams  : {bigrams}\n"
    "Top NMF terms: {nmf_terms}\n"
    "Representative project tokens:\n"
    "{snippets}\n\n"

    "Instructions:\n"
    "- proposed_label must be a short canonical noun phrase, usually 3 to 7 words.\n"
    "- description must be exactly one sentence in plain English.\n"
    "- Write description in this pattern when supported by the evidence: "
    "Requests for [mechanism/request] are used to [classroom function/use case] "
    "for [population or setting], often framed as [plain-English framing] when clearly present.\n"
    "- Prefer concrete mechanism and classroom function over broad category language.\n"
    "- Translate any framing/category/subcategory token signals into normal language; never copy raw token strings.\n"
    "- If the topic appears broad, identify the narrower mechanism, use case, or population "
    "if the evidence supports it.\n"
    "- Use coherence_flag='mixed' only when no single dominant theme clearly leads.\n"
    "- If one theme is primary and another is secondary, mark the topic coherent.\n"
    "- notes should usually be empty; use notes only for mixed, redundant, or unclear topics.\n"
    "- Do not write donor-facing insights, implications, or rhetorical flourishes.\n\n"

    "Output requirements:\n"
    "- proposed_label: short and stable\n"
    "- description: exactly one dense grounded sentence\n"
    '- coherence_flag: one of "coherent", "mixed", "redundant", "unclear"\n'
    "- notes: usually empty, unless needed for edge cases\n\n"

    f"Return a JSON object with exactly these keys:\n"
    f"{GROUPBY_FIELD}, topic_id, proposed_label, description, coherence_flag, notes"
)

# Build lightweight snippet text from the top projects attached to each topic.
pid_text = df.set_index("project_id")["tokens"].apply(lambda t: " ".join(t[:40]))
topic_inputs = [
    build_input(
        t,
        weights_df=weights_df,
        pid_text=pid_text,
        groupby_field=GROUPBY_FIELD,
        n_representative=N_REPRESENTATIVE,
        top_terms_in_prompt=TOP_TERMS_IN_PROMPT,
    )
    for _, t in topics_df.iterrows()
]

topic_order = {
    (_norm_group_value(inp["group_value"]), int(inp["topic_id"])): i
    for i, inp in enumerate(topic_inputs)
}

results = []
with ThreadPoolExecutor(max_workers=LABELING_MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _label_with_retry,
            inp,
            client=client,
            model_labeling=MODEL_LABELING,
            system_prompt=SYSTEM,
            user_prompt_template=PROMPT,
            groupby_field=GROUPBY_FIELD,
            warnings_path=WARNINGS_PATH,
            max_retries=MAX_RETRIES,
        ): inp
        for inp in topic_inputs
    }
    for future in as_completed(futures):
        inp = futures[future]
        obj = future.result()
        results.append(obj)
        print(f"  {inp['group_value']} / topic {inp['topic_id']} → {obj.get('proposed_label', '?')}")

# Re-sort to the original topic order so downstream joins stay stable.
bad_sort_keys = []
for r in results:
    norm_key = (
        _norm_group_value(r.get(GROUPBY_FIELD, "")),
        _safe_topic_id(r.get("topic_id", -1)),
    )
    if norm_key not in topic_order:
        bad_sort_keys.append({"group": r.get(GROUPBY_FIELD, ""), "topic_id": r.get("topic_id", "")})

if bad_sort_keys:
    raise ValueError(
        f"LABELING_SORT_KEY_MISMATCH: {len(bad_sort_keys)} label result(s) did not match the input topic order. "
        f"Examples: {bad_sort_keys[:5]}"
    )

results = sorted(
    results,
    key=lambda r: topic_order[
        (_norm_group_value(r.get(GROUPBY_FIELD, "")), _safe_topic_id(r.get("topic_id", -1)))
    ],
)

with open(OUT("analysis", "llm_topic_labels.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)

print(f"\n{len(results)} labels saved")

  Art Supplies / topic 5 → Mixed media painting and drawing materials
  Art Supplies / topic 0 → STEAM science and engineering models
  Art Supplies / topic 7 → Art supply requests for self-expression
  Art Supplies / topic 6 → Lunar and holiday culture crafts
  Art Supplies / topic 2 → Fine motor playdough and clay practice
  Art Supplies / topic 1 → Critical thinking art and craft supplies
  Art Supplies / topic 4 → Basic drawing and dry erase supplies
  Art Supplies / topic 9 → Cricut and vinyl apparel customization
  Art Supplies / topic 10 → Kindness card and bracelet service
  Art Supplies / topic 11 → Art supply requests for classroom comfort
  Art Supplies / topic 8 → Art materials for confidence building
  Art Supplies / topic 13 → Behavior incentive reward materials
  Art Supplies / topic 18 → Glue stick and construction paper supplies
  Art Supplies / topic 3 → Stress regulation sensory calm tools
  Art Supplies / topic 15 → Bulletin board paper for welcoming displays
  Art 

In [9]:
# ── Label post-processing ──────────────────────────────────────────────────
# Keep only parseable labeling results and validate them against source topics.

parse_errors = [r for r in results if r.get("parse_error")]
if parse_errors:
    print(f"WARNING: {len(parse_errors)} topics failed JSON parse — excluded from synthesis:")
    for e in parse_errors:
        print(f"  {e.get(GROUPBY_FIELD, '?')} / topic {e.get('topic_id', '?')}")

labels_df = pd.DataFrame([r for r in results if not r.get("parse_error")])

assert GROUPBY_FIELD in labels_df.columns, (
    f"labels_df missing '{GROUPBY_FIELD}' — re-run labeling for this groupby field"
)

# Source-of-truth allowed groups come from the source topic table, not model output.
ALLOWED_GROUP_VALUES = [
    str(g)
    for g in topics_df[GROUPBY_FIELD].dropna().drop_duplicates().tolist()
]

invalid_groups = sorted(
    set(labels_df[GROUPBY_FIELD].astype(str).unique()) - set(ALLOWED_GROUP_VALUES)
)
if invalid_groups:
    raise ValueError(
        f"Invalid {GROUPBY_FIELD} values returned during labeling: {invalid_groups}"
    )

# Normalize types after validation so later joins are stable.
labels_df[GROUPBY_FIELD] = labels_df[GROUPBY_FIELD].astype(str)
labels_df["topic_id"] = labels_df["topic_id"].astype(int)

labeled_groups = set(labels_df[GROUPBY_FIELD].unique()) if not labels_df.empty else set()
LABELING_FAILED_GROUPS = sorted(set(ALLOWED_GROUP_VALUES) - labeled_groups)

labels_df.groupby("coherence_flag").size().rename("count").to_frame()
labels_df[[GROUPBY_FIELD, "topic_id", "proposed_label", "coherence_flag", "description"]]

,project_category,topic_id,proposed_label,coherence_flag,description
0,Art Supplies,0,STEAM science and engineering models,coherent,Requests for science and engineering materials...
1,Art Supplies,1,Critical thinking art and craft supplies,coherent,Requests for art and craft materials to suppor...
2,Art Supplies,2,Fine motor playdough and clay practice,coherent,Requests for fine-motor skill development thro...
3,Art Supplies,3,Stress regulation sensory calm tools,coherent,Requests for calming sensory self-regulation i...
4,Art Supplies,4,Basic drawing and dry erase supplies,coherent,"Requests for drawing tools like pencils, marke..."
...,...,...,...,...,...
253,Virtual Trips,13,National Board Certification support,coherent,Requests to pursue National Board Certificatio...
254,Virtual Visitors,0,Virtual jazz coaching videos,coherent,Requests for video instruction and personalize...
255,Virtual Visitors,1,Virtual cooperative adventure visits,coherent,Requests for virtual cooperative adventure vis...
256,Virtual Visitors,2,Virtual author and book visits,coherent,Requests to invite authors for live book talks...


---
## Step 6 — Synthesis

Produces two synthesis passes in sequence:

1. **Cross-group synthesis** — a single LLM call over all topic lines to identify
   patterns, contrasts, framing logic, and notable boundaries across the corpus.
2. **Per-group synthesis** — one LLM call per group, run in parallel, to surface
   each group's dominant internal mechanisms.

Both passes produce plain text saved to `analysis/llm_synthesis_*.txt` and are
assembled in Step 7 as the evidence base for the structured JSON insight call.

In [10]:
# ── SYNTHESIS — cross-group + per-group loops ─────────────────────────────
# First synthesize the whole landscape, then synthesize each group in parallel.

SYNTHESIS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "Your job is to synthesize topic-level evidence into stable, decision-useful analytic findings "
    "for internal strategy work. You are not writing a polished memo and you are not doing creative interpretation. "
    "You are performing disciplined evidence grouping.\n\n"

    "Core objective:\n"
    "Produce findings that are specific, repeatable, and tightly grounded in the supplied topic lines. "
    "Literal evidence discipline is more important than elegance.\n\n"

    "Evidence rules:\n"
    "- Treat the supplied topic lines as the full evidence base.\n"
    "- Every finding must be supported by explicitly named topics.\n"
    "- Use only topics that directly support the claim.\n"
    "- Do not generalize beyond the named supporting topics.\n"
    "- If evidence is borderline, omit the finding rather than stretching it.\n\n"

    "Merge rules:\n"
    "- Merge topics only when they share the same dominant mechanism or classroom function.\n"
    "- Do not merge topics just because they involve the same product category, school setting, or broad theme.\n"
    "- Keep access/accommodation, regulation/behavior, maintenance/operations, identity/belonging, and routine/workflow "
    "separate unless the evidence clearly shows they are the same pattern.\n"
    "- If one candidate finding is broader and another is a more specific mechanism-level version, prefer the more specific one.\n\n"

    "Stability rules:\n"
    "- Review topics in the order given.\n"
    "- Prefer narrower mechanism-level patterns over broad umbrella summaries.\n"
    "- Use the same output structure every time.\n"
    "- If two formulations are both defensible, choose the more literal and reusable one.\n\n"

    "Support-citation rules:\n"
    "- For every finding, include a 'Supporting topics' line.\n"
    "- Each supporting topic must be written exactly as <group_value>|<topic_id>.\n"
    "- Example: Books|7\n"
    "- Do not write labels or prose in place of the support references.\n\n"

    "Writing rules:\n"
    "- Be precise, concrete, and analytical.\n"
    "- Do not mention NMF, model, cluster, pipeline, or analytical machinery.\n"
    "- Do not write donor-facing rhetoric.\n"
    "- Do not use vague abstractions when the evidence supports a concrete mechanism.\n"
    "- Do not invent a cleaner pattern than the evidence supports."
)

topic_lines = build_topic_lines(
    labels_df,
    GROUPBY_FIELD,
    top_terms_count=SYNTHESIS_TOP_TERMS_COUNT,
)

SYNTHESIS_PROMPT_CROSS_GROUP = f"""
Below is a list of topic lines discovered from teacher project request essays on DonorsChoose,
grouped by "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

Each line contains:
- group value
- topic number
- topic label
- coherence flag
- top terms
- one-sentence description

Topic lines:
{topic_lines}

Your task:
Synthesize the topic lines into a stable cross-group analysis that will later be used to generate
external-facing insights. Preserve nuance, but be disciplined about grouping.

Follow this exact workflow:
1. Review the topic lines in the order given.
2. Identify candidate cross-group patterns only when multiple topics share the same dominant mechanism,
   classroom function, or hidden operating logic.
3. Keep distinct mechanisms separate even when they live in similar product categories.
4. Rank findings by:
   a. breadth across groups
   b. specificity of the shared mechanism
   c. clarity of distinction from nearby themes
5. Omit weak or borderline patterns rather than padding.

Return plain text only, using exactly this structure and these section headings:

CROSS-GROUP PATTERNS
Finding 1
Title: ...
Pattern: ...
Why this is distinct: ...
Supporting topics: GroupA|1; GroupB|4; GroupC|7

Finding 2
Title: ...
Pattern: ...
Why this is distinct: ...
Supporting topics: ...

IMPORTANT CONTRASTS
Finding 1
Title: ...
Contrast: ...
Why the distinction matters: ...
Supporting topics: ...

Finding 2
Title: ...
Contrast: ...
Why the distinction matters: ...
Supporting topics: ...

FRAMING AND PITCH STRATEGIES
Finding 1
Title: ...
Pattern: ...
Why it matters for interpretation: ...
Supporting topics: ...

Finding 2
Title: ...
Pattern: ...
Why it matters for interpretation: ...
Supporting topics: ...

NOTABLE ABSENCES OR BOUNDARIES
Finding 1
Title: ...
Boundary: ...
Why this boundary matters: ...
Supporting topics: ...

Hard requirements:
- CROSS-GROUP PATTERNS: return 5 to 8 findings
- IMPORTANT CONTRASTS: return 2 to 4 findings
- FRAMING AND PITCH STRATEGIES: return 1 to 3 findings
- NOTABLE ABSENCES OR BOUNDARIES: return 0 to 2 findings
- Every finding must include a Supporting topics line
- Supporting topics must use exact <group_value>|<topic_id> format
- Do not use bullet points
- Do not skip section headings
- Do not repeat the same pattern in multiple sections
- Do not merge findings just because they sound similar at a high level
- Prefer mechanism-level explanations over broad summaries like “access,” “engagement,” or “support”
- If a pattern is supported by only one group, it does not belong in CROSS-GROUP PATTERNS
- If a distinction is real but narrow, put it in IMPORTANT CONTRASTS rather than inflating it into a cross-group signature

Write like a rigorous internal analyst, not like a speechwriter.
""".strip()

PER_GROUP_INSTRUCTIONS = """
Your task:
Identify the strongest, most decision-useful patterns within this single group.

Follow this exact workflow:
1. Review the topic lines in the order given.
2. Identify the dominant subthemes in the group.
3. Separate nearby themes unless they clearly share the same dominant mechanism or classroom function.
4. Prefer narrower mechanism-level findings over broad category summaries.
5. Omit weak or redundant findings rather than padding.

Return plain text only, using exactly this structure and these section headings:

CORE SUBTHEMES
Finding 1
Title: ...
Pattern: ...
Why this is distinct: ...
Supporting topics: <group_value>|<topic_id>; <group_value>|<topic_id>

Finding 2
Title: ...
Pattern: ...
Why this is distinct: ...
Supporting topics: ...

IMPORTANT INTERNAL SPLITS
Finding 1
Title: ...
Contrast: ...
Why the distinction matters: ...
Supporting topics: ...

Finding 2
Title: ...
Contrast: ...
Why the distinction matters: ...
Supporting topics: ...

FRAMING AND PITCH LOGIC
Finding 1
Title: ...
Pattern: ...
Why it matters for interpretation: ...
Supporting topics: ...

BOUNDARIES OR NON-CENTRAL SIGNALS
Finding 1
Title: ...
Boundary: ...
Why this boundary matters: ...
Supporting topics: ...

Hard requirements:
- CORE SUBTHEMES: return 3 to 6 findings
- IMPORTANT INTERNAL SPLITS: return 1 to 3 findings
- FRAMING AND PITCH LOGIC: return 1 to 3 findings
- BOUNDARIES OR NON-CENTRAL SIGNALS: return 0 to 2 findings
- Every finding must include a Supporting topics line
- Supporting topics must use exact <group_value>|<topic_id> format
- Do not use bullet points
- Do not skip section headings
- Do not repeat the same idea across multiple sections
- Use 'mixed' topics carefully; do not let one mixed topic dominate a whole finding unless the collision itself is the finding
- If one theme is primary and another is secondary, keep the finding centered on the primary theme
- Do not give generic summaries of the group label; surface mechanism-level patterns inside the group

Write like a rigorous internal analyst, not like a speechwriter.
""".strip()

synthesis_cross = _call_with_retry(
    SYNTHESIS_PROMPT_CROSS_GROUP,
    client=client,
    model_name=MODEL_SYNTHESIS,
    system_prompt=SYNTHESIS_SYSTEM,
    max_retries=MAX_RETRIES,
)
if synthesis_cross is None:
    append_warning(
        WARNINGS_PATH,
        "03_insights_generation",
        "SYNTHESIS_CROSS_GROUP_FAILED",
        "Cross-group synthesis failed",
        context={},
    )
    raise RuntimeError("Cross-group synthesis failed; stopping before downstream cells.")

with open(OUT("analysis", "llm_synthesis_cross_group.txt"), "w", encoding="utf-8") as f:
    f.write(synthesis_cross)

# Excluded groups were removed from df in the Parameters cell; they cannot
# appear in ALLOWED_GROUP_VALUES. Filter only for label coverage.
groups = [
    g for g in ALLOWED_GROUP_VALUES
    if g in set(labels_df[GROUPBY_FIELD].unique())
]

per_group_results = {}
SYNTHESIS_FAILED_GROUPS = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            synthesize_one_group,
            g,
            labels_df=labels_df,
            groupby_field=GROUPBY_FIELD,
            group_description=GROUP_DESCRIPTION,
            per_group_instructions=PER_GROUP_INSTRUCTIONS,
            client=client,
            model_name=MODEL_SYNTHESIS,
            system_prompt=SYNTHESIS_SYSTEM,
            warnings_path=WARNINGS_PATH,
            outpath_func=OUT,
            max_retries=MAX_RETRIES,
        ): g
        for g in groups
    }
    for future in as_completed(futures):
        submitted_group = futures[future]
        try:
            group, result = future.result()
        except Exception as e:
            SYNTHESIS_FAILED_GROUPS.append(str(submitted_group))
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "SYNTHESIS_GROUP_FAILED",
                f"Synthesis failed for group '{submitted_group}'",
                context={"group": submitted_group, "error": str(e)},
            )
            print(f"FAILED: {submitted_group} | error: {e}")
            continue

        if result is not None:
            per_group_results[group] = result
            print(f"Done: {group}")
        else:
            SYNTHESIS_FAILED_GROUPS.append(str(group))
            print(f"FAILED: {group}")

# Preserve the original group order before handing results downstream.
per_group_results = {g: per_group_results[g] for g in groups if g in per_group_results}

if not per_group_results:
    raise RuntimeError("No per-group synthesis results were produced; stopping before downstream cells.")

Done: Computers & Tablets
Done: Food, Clothing & Hygiene
Done: Instructional Technology
Done: Books
Done: Educational Kits & Games
Done: Flexible Seating
Done: Art Supplies
Done: Classroom Basics
Done: Virtual Visitors
Done: Lab Equipment
Done: Musical Instruments
Done: Virtual Trips
Done: Sports & Exercise Equipment
Done: Reading Nooks, Desks & Storage


---
## Step 7 — Structured Insight Extraction

Assembles the cross-group and per-group synthesis texts and asks the model to
return a structured JSON object with `key_insights` (cross-group) and `by_group`
(per-group, one list per group value). The response is validated, normalized,
and written to `insights_candidates.json` as a pre-verification snapshot.

In [11]:
# ── EXTERNAL-FACING INSIGHTS — structured JSON call ───────────────────────
# Ask the model for a single JSON object, then normalize and validate it.

OUTPUT_GROUP_KEY = "by_group"
REQUIRED_GROUP_VALUES = list(per_group_results.keys())

if not REQUIRED_GROUP_VALUES:
    raise RuntimeError("No synthesized group results are available; cannot build external-facing insights.")

synthesis_parts = []
if synthesis_cross:
    synthesis_parts.append(f"=== CROSS-GROUP ANALYSIS ===\n{synthesis_cross}")
for group_value, text in per_group_results.items():
    if text:
        synthesis_parts.append(f"=== {group_value} ===\n{text}")
synthesis_input = "\n\n".join(synthesis_parts)

INSIGHTS_SYSTEM = (
    "You are a senior program analyst at an educational nonprofit. "
    "You turn structured internal analysis into clear, decision-facing insights for external audiences: "
    "foundations, corporate partners, policymakers, executives, and major donors. "
    "You return ONLY valid JSON. No preamble, no explanation, no markdown fences.\n\n"

    "You are working from a structured synthesis that already groups evidence into findings, contrasts, "
    "framing logic, and boundaries, and that includes explicit Supporting topics lines. "
    "Treat those synthesized findings as the evidence base.\n\n"

    "Core objective:\n"
    "- Surface the few patterns that would genuinely change how a thoughtful outsider interprets teacher demand.\n"
    "- Prefer mechanism-level reveals over broad thematic summaries.\n"
    "- Preserve evidence discipline: every insight must be grounded in the strongest matching supporting topics.\n"
    "- Write for intelligent outsiders who are not close to classroom realities.\n\n"

    "Evidence rules:\n"
    "- Use the structured synthesis as the primary evidence source.\n"
    "- Prefer source_topics that are explicitly named in Supporting topics lines.\n"
    "- Do not infer broad support if the synthesis only gives narrow support.\n"
    "- Prefer fewer well-matched source_topics over padding with weak ones.\n"
    "- Do not merge adjacent synthesized findings unless they clearly support the same external-facing insight.\n\n"

    "Key-insight distinctness rules:\n"
    "- Key insights must be mutually distinct.\n"
    "- Two key insights are not distinct if they lead to the same outsider takeaway, the same hidden mechanism, "
    "or the same funding/strategy implication.\n"
    "- If two candidate key insights overlap materially, keep the stronger one or merge them into one sharper insight.\n"
    "- Prefer the smallest non-overlapping set of key insights that fully captures the strongest cross-group patterns.\n"
    "- Do not split one broad pattern into multiple final insights unless each split produces a meaningfully different "
    "outsider interpretation or different action implication.\n\n"

    "Selection tie-break rules:\n"
    "- When choosing between two plausible candidate insights, prefer the one with:\n"
    "  1. broader support across synthesized findings,\n"
    "  2. clearer mechanism-level distinctness,\n"
    "  3. stronger outsider takeaway,\n"
    "  4. fewer but stronger source_topics,\n"
    "  5. less overlap with already selected insights.\n\n"

    "By-group rules:\n"
    "- For each group, first identify the dominant internal mechanisms.\n"
    "- Only add another by-group insight if it is not a subcase or restatement of one already selected.\n"
    "- Prefer coverage of the group's main internal story over exhaustive enumeration.\n\n"

    "Audience and style rules:\n"
    "- Write for intelligent outsiders who are not close to classroom realities.\n"
    "- The insight should feel like a reveal, not like a category recap.\n"
    "- Focus on what teachers are actually doing, compensating for, or signaling through requests.\n"
    "- Be direct, concrete, and human.\n"
    "- Not academic.\n"
    "- Not corporate boilerplate.\n"
    "- Not donor-marketing fluff.\n"
    "- No mention of topics, NMF, models, clusters, prompts, or analytical machinery.\n"
    "- No hedging such as 'may suggest' or 'could indicate'.\n"
    "- No filler.\n\n"

    "Output rules:\n"
    "- Return ONLY valid JSON.\n"
    "- Use exactly the required schema and fields.\n"
    "- Do not add extra fields.\n"
    "- Do not include markdown fences, commentary, or explanation outside the JSON object."
)

example_source_topics = []
example_rows = (
    labels_df[[GROUPBY_FIELD, "topic_id"]]
    .drop_duplicates()
    .head(3)
    .to_dict("records")
)

for row in example_rows:
    example_source_topics.append(f"{row[GROUPBY_FIELD]}|{int(row['topic_id'])}")

if not example_source_topics:
    for i, gv in enumerate(REQUIRED_GROUP_VALUES[:2], start=1):
        example_source_topics.append(f"{gv}|{i}")

example_source_topics_json = json.dumps(example_source_topics, ensure_ascii=False)

INSIGHTS_PROMPT = f"""
You are given a structured synthesis of classroom project request patterns.

The synthesis is grouped by the field "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

It contains:
- cross-group findings
- within-group findings
- important contrasts
- framing and pitch logic
- boundaries or non-central signals
- explicit Supporting topics lines in the format <group_value>|<topic_id>

Structured synthesis:
{synthesis_input}

Your task:
Produce a decision-facing insights document for external audiences —
foundation leaders, corporate partners, policymakers, executives, and major donors —
who are intelligent but not close to classroom realities.

The best insights should make a reader think:
"I would not have seen that from surface-level demand, and this changes how I interpret what classrooms need."

WORKING METHOD:
1. Read the structured synthesis as a set of evidence units, not just as prose.
2. Identify all plausible candidate cross-group insights.
3. Remove or merge any candidate whose core implication materially overlaps another candidate.
4. Keep only the strongest non-overlapping set.
5. Then generate the final JSON.

SELECTION CRITERIA:
- Prioritize insights that reveal hidden operating pressures, misread demand, underestimated classroom functions,
  invisible tradeoffs, or shifts in how teachers are compensating for unmet needs.
- Prefer insights that would matter for funding strategy, partnership design, policy interpretation,
  or external messaging.
- Avoid observations that merely restate what the group label already implies.
- Avoid methodological commentary and data-practitioner-only insights.
- Prefer mechanism-level reveals over broad summary language.

KEY-INSIGHT DISTINCTNESS RULE:
- Each key insight must have a distinct core implication.
- Before finalizing "key_insights", check whether any two candidate key insights share the same hidden mechanism,
  same outsider takeaway, or same funding/strategy implication.
- If they do, keep the stronger one or merge them into one sharper insight.
- Do not return multiple key insights that are different phrasings or slight extensions of the same underlying point.

MERGE RULE:
- Combine synthesized findings into one final insight only when they clearly support the same external-facing interpretation of teacher demand.
- Shared product domain, broad similarity, or adjacent classroom context is not enough by itself.
- If a broader candidate and a narrower, more decision-useful candidate are both available, prefer the narrower one.
- Do not split one broad pattern into multiple final insights unless each split changes what an outsider should conclude.

COUNT + COVERAGE REQUIREMENTS:
- Return between {CROSS_MIN_INSIGHTS} and {CROSS_MAX_INSIGHTS} cross-group insights in "key_insights".
- Treat {CROSS_MIN_INSIGHTS} as the default target.
- Start by asking whether the strongest {CROSS_MIN_INSIGHTS} key insights already capture the cross-group story.
- Only add more than {CROSS_MIN_INSIGHTS} if an additional insight is clearly distinct, non-overlapping, strongly supported,
  and would materially change what an outside reader learns.
- Never add an insight just because room remains before {CROSS_MAX_INSIGHTS}.
- Prefer the smallest non-overlapping set within the allowed range that fully captures the strongest cross-group patterns.

- "{OUTPUT_GROUP_KEY}" must include every group value in this exact list:
  {json.dumps(REQUIRED_GROUP_VALUES, ensure_ascii=False)}

- Return between {PER_GROUP_MIN_INSIGHTS} and {PER_GROUP_MAX_INSIGHTS} strongest defensible insights for each group value above.
- Treat {PER_GROUP_MIN_INSIGHTS} as the default target for each group.
- Start by asking whether the strongest {PER_GROUP_MIN_INSIGHTS} insights already capture the group's main internal story.
- Only add more than {PER_GROUP_MIN_INSIGHTS} when the additional insight introduces a clearly different mechanism,
  not just a narrower example, subcase, or rephrasing.
- Do not omit group values silently.
- Prefer slightly weaker but still defensible coverage over omission.
- Keep by-group insights focused on the group's dominant internal mechanisms and avoid repeating the same implication in multiple phrasings.

SELECTION TIE-BREAK RULES:
- When choosing between two plausible candidate insights, prefer the one with:
  1. broader support across synthesized findings,
  2. clearer mechanism-level distinctness,
  3. stronger outsider takeaway,
  4. fewer but stronger source_topics,
  5. less overlap with already selected insights.

INSIGHT STRUCTURE:
For every insight, use exactly these fields:

- title:
  a crisp declarative headline that feels like a reveal, not a topic label

- what_seeing:
  2-4 sentences in plain, concrete language describing what teachers are actually doing, asking for,
  compensating for, or signaling through requests.
  Write as an outsider-facing interpretation of the evidence, not as a summary of an analysis process.

- why_easy_to_miss:
  1-2 sentences explaining why this pattern is hard to see from conventional reporting,
  surface-level category labels, or simplistic interpretations of teacher demand.

- source_topics:
  a list of the strongest grounding topics for the insight.
  Use only topics that directly support the claim.
  Usually 2-4 topics is enough, but do not force a fixed count.
  Prefer fewer strong matches over weak padding.
  Required format:
    - each item must be a string
    - each string must be exactly "<group_value>|<topic_id>"
    - group_value must exactly match one of the group values shown in the synthesis
    - topic_id must be the integer topic number for that group
  Example for this run: {example_source_topics_json}

SOURCE_TOPIC RULES:
- Prefer source_topics that are explicitly named in Supporting topics lines.
- If a synthesized finding names multiple supporting topics but only some directly support your final claim,
  include only the strongest matches.
- If a topic is illustrative but not necessary to support the claim, leave it out.
- Do not use labels, prose, section names, or invented group names in place of group_value.
- source_topics must be a list of strings only, not objects.
- Do not include any extra fields beyond: title, what_seeing, why_easy_to_miss, source_topics.

STYLE:
- Direct, confident, and human.
- Not academic.
- Not corporate boilerplate.
- Not donor-marketing fluff.
- No mention of topics, models, clusters, or analytical machinery.
- No hedging phrases like "may suggest" or "could indicate".
- No filler.

VALIDITY RULES:
- Return valid JSON only.
- The response is incomplete if any required group value is missing from "{OUTPUT_GROUP_KEY}".
- Do not return markdown fences or explanatory text.
- Every source_topics item must follow the exact required format.
- Every insight object must contain exactly these four fields:
  title, what_seeing, why_easy_to_miss, source_topics

Return a JSON object with exactly this structure:
{{
  "key_insights": [/* {CROSS_MIN_INSIGHTS}-{CROSS_MAX_INSIGHTS} insight objects */],
  "{OUTPUT_GROUP_KEY}": {{
    "<group value>": [/* {PER_GROUP_MIN_INSIGHTS}-{PER_GROUP_MAX_INSIGHTS} insight objects */],
    ...
  }}
}}
""".strip()

resp = client.chat.completions.create(
    model=MODEL_SYNTHESIS,
    messages=[
        {"role": "system", "content": INSIGHTS_SYSTEM},
        {"role": "user", "content": INSIGHTS_PROMPT},
    ],
    response_format={"type": "json_object"},
)

raw_json = strip_json_fences(resp.choices[0].message.content)
insights_data = json.loads(raw_json)

if not isinstance(insights_data, dict):
    raise ValueError("Top-level insights response is not a JSON object.")

raw_key = insights_data.get("key_insights", [])
raw_by_group = insights_data.get(OUTPUT_GROUP_KEY, {})

if not isinstance(raw_key, list):
    raise ValueError("key_insights is not a list.")
if not isinstance(raw_by_group, dict):
    raise ValueError(f"{OUTPUT_GROUP_KEY} is not an object.")

normalized = {
    "key_insights": [
        normalize_insight(i, required_group_values=REQUIRED_GROUP_VALUES)
        for i in raw_key
    ],
    OUTPUT_GROUP_KEY: {},
}

missing_group_values = [g for g in REQUIRED_GROUP_VALUES if g not in raw_by_group]
if missing_group_values:
    raise ValueError(f"Model omitted required group values: {missing_group_values}")

extra_group_values = [g for g in raw_by_group.keys() if g not in REQUIRED_GROUP_VALUES]
if extra_group_values:
    print(f"WARNING: extra group values returned and ignored: {extra_group_values}")

for group_value in REQUIRED_GROUP_VALUES:
    items = raw_by_group.get(group_value, [])
    if not isinstance(items, list):
        raise ValueError(f"{OUTPUT_GROUP_KEY}['{group_value}'] is not a list.")
    normalized[OUTPUT_GROUP_KEY][group_value] = [
        normalize_insight(i, required_group_values=REQUIRED_GROUP_VALUES)
        for i in items
    ]

insights_data = normalized

if not (CROSS_MIN_INSIGHTS <= len(insights_data["key_insights"]) <= CROSS_MAX_INSIGHTS):
    raise ValueError(
        f"Expected between {CROSS_MIN_INSIGHTS} and {CROSS_MAX_INSIGHTS} key insights, "
        f"got {len(insights_data['key_insights'])}"
    )

bad_group_counts = {
    group_value: len(items)
    for group_value, items in insights_data[OUTPUT_GROUP_KEY].items()
    if not (PER_GROUP_MIN_INSIGHTS <= len(items) <= PER_GROUP_MAX_INSIGHTS)
}
if bad_group_counts:
    raise ValueError(
        f"Expected between {PER_GROUP_MIN_INSIGHTS} and {PER_GROUP_MAX_INSIGHTS} insights per group value, "
        f"got: {bad_group_counts}"
    )

insights_candidates_for_save = {
    "key_insights": [
        project_insight_for_saved_candidates(i)
        for i in insights_data["key_insights"]
    ],
    OUTPUT_GROUP_KEY: {
        group_value: [
            project_insight_for_saved_candidates(i)
            for i in items
        ]
        for group_value, items in insights_data[OUTPUT_GROUP_KEY].items()
    },
}

# Mid-cell write required by spec: post-normalize_insight(), pre-verification.
write_json(OUT("insights", "insights_candidates.json"), insights_candidates_for_save)

PosixPath('/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/project_category/2026-04-10/20260410_111800_project_category_77e319db/insights/insights_candidates.json')

---
## Step 8 — Topic Verification

Verifies that each claimed source topic genuinely supports its insight. Topics
that are too broad, indirect, or adjacent are dropped.

Verification strategy:
- **Cross-group insights** — always verified.
- **By-group insights** — verified only when they cite ≥ `by_group_min_source_topics`
  source topics; narrow local findings with 1–2 cited topics pass through without
  an additional API call.

In [12]:
# ── Topic verification ─────────────────────────────────────────────────────
# Verify source-topic grounding for final insights.
# Strategy:
# - always verify key_insights
# - verify by-group insights only when they cite 3+ source topics
#   (broad claims are more likely to overclaim support than narrow local ones)

VERIFY_SYSTEM = (
    "You are validating whether cited topics directly support an insight. "
    "Be strict. Keep a topic only if its label and description directly support the claim. "
    "If support is broad, indirect, adjacent, or only loosely related, drop it. "
    "Prefer false negatives to false positives. "
    "Return only valid JSON."
)

VERIFY_BY_GROUP_MIN_SOURCE_TOPICS = VERIFY_CFG["by_group_min_source_topics"]

# Always verify cross-group/key insights
insights_data["key_insights"], key_verify_stats = _verify_insight_list(
    insights_data.get("key_insights", []),
    labels_df=labels_df,
    groupby_field=GROUPBY_FIELD,
    required_group_values=REQUIRED_GROUP_VALUES,
    client=client,
    model_verify=MODEL_VERIFY,
    system_prompt=VERIFY_SYSTEM,
    warnings_path=WARNINGS_PATH,
    min_source_topics_to_verify=1,
)

group_verify_stats = {}
for cat in insights_data.get(OUTPUT_GROUP_KEY, {}):
    verified_items, stats = _verify_insight_list(
        insights_data[OUTPUT_GROUP_KEY][cat],
        labels_df=labels_df,
        groupby_field=GROUPBY_FIELD,
        required_group_values=REQUIRED_GROUP_VALUES,
        client=client,
        model_verify=MODEL_VERIFY,
        system_prompt=VERIFY_SYSTEM,
        warnings_path=WARNINGS_PATH,
        min_source_topics_to_verify=VERIFY_BY_GROUP_MIN_SOURCE_TOPICS,
    )
    insights_data[OUTPUT_GROUP_KEY][cat] = verified_items
    group_verify_stats[cat] = stats

total_group_changed = sum(v["changed_count"] for v in group_verify_stats.values())
total_group_zeroed = sum(v["dropped_to_zero_count"] for v in group_verify_stats.values())
total_group_topics_before = sum(v["topics_before"] for v in group_verify_stats.values())
total_group_topics_after = sum(v["topics_after"] for v in group_verify_stats.values())

print("Verification complete.")
print(
    f"Key insights: {key_verify_stats['insight_count']} insights | "
    f"{key_verify_stats['changed_count']} changed | "
    f"{key_verify_stats['dropped_to_zero_count']} dropped to zero topics | "
    f"{key_verify_stats['topics_before']} -> {key_verify_stats['topics_after']} source topics"
)
print(
    f"By-group insights: {sum(v['insight_count'] for v in group_verify_stats.values())} insights | "
    f"{total_group_changed} changed | "
    f"{total_group_zeroed} dropped to zero topics | "
    f"{total_group_topics_before} -> {total_group_topics_after} source topics | "
    f"verified only when source_topics >= {VERIFY_BY_GROUP_MIN_SOURCE_TOPICS}"
)

Adjusted source_topics: Many requests are really for participation infrastructure, not classroom extras | 4 -> 3
Adjusted source_topics: Teachers are using movement and seating as behavior support built into instructi | 4 -> 3
Adjusted source_topics: Basic needs show up in requests as academic readiness problems, not separate cha | 4 -> 3
Adjusted source_topics: Not all behavior-related requests mean the same thing | 4 -> 2
Adjusted source_topics: Art requests often keep basic participation alive, not launch special projects | 4 -> 3
Adjusted source_topics: Some art materials are being used as regulation tools rather than art tools | 3 -> 2
Adjusted source_topics: The biggest basic-supplies story is operational continuity | 4 -> 2
Adjusted source_topics: Whiteboards are being requested as an instructional system, not just a supply it | 3 -> 2
Adjusted source_topics: Hands-on kit demand is strongest where teachers need students to physically buil | 4 -> 0
Adjusted source_topics: Flexibl

---
## Step 9 — Evidence Tables

Expands each verified insight into a flat summary row (`insights_flat.csv`) and
a per-topic support detail row (`insight_topic_support.csv`). Projects are ranked
by combined topic_share across all of an insight's verified topics so Looker
links surface the most representative essays.

In [13]:
# ── VERIFIED INSIGHT SUPPORT TABLES ────────────────────────────────────────

assert "project_topic_bridge_df" in dir() and not project_topic_bridge_df.empty, (
    "project_topic_bridge_df missing — ensure the bridge-build cell ran successfully"
)

BRIDGE_LOOKUP = build_bridge_lookup(project_topic_bridge_df)

label_index = build_label_index(
    labels_df,
    groupby_field=GROUPBY_FIELD,
    warnings_path=WARNINGS_PATH,
)

insights_flat_df, insight_topic_support_df = build_verified_insight_tables(
    insights_data,
    OUTPUT_GROUP_KEY,
    groupby_field=GROUPBY_FIELD,
    bridge_lookup=BRIDGE_LOOKUP,
    label_index=label_index,
    run_id=RUN_ID,
    # csv_max_ids_per_insight controls insight row density; see params.yaml.
    top_project_id_limit=CSV_MAX_IDS_PER_INSIGHT,
)

insights_flat_df.to_csv(OUT("insights", "insights_flat.csv"), index=False)
insight_topic_support_df.to_csv(
    OUT("insights", "insight_topic_support_candidates.csv"),
    index=False,
)

print(f"Candidate insights summarized: {len(insights_flat_df):,}")

Candidate insights summarized: 47


---
## Step 10 — Packaging, Dedupe & Topline Selection

Three sequential operations:

1. **Packaging** — apply quality threshold gates. Insights that don't clear
   `min_verified_topic_count`, `min_supporting_project_count`, and
   `min_mean_topic_share` are rejected.
2. **Deduplication** — deterministic overlap-based dedup within pair types.
3. **Topline selection** — assigns `report_section` to each accepted insight:
   `main_cross` (top N by quality rank), `main_by_group` (top 1 per group),
   `appendix_cross`, `appendix_by_group`.

In [16]:
# ── PACKAGING + DEDUPE + SIMPLE TOPLINE CUT ────────────────────────────────
# Business rule:
# - accept insights that clear threshold gates
# - remove obvious duplicates
# - then choose topline from the remaining pool
#
# Main section:
# - top N cross-category insights by:
#     1) supporting_project_count
#     2) verified_topic_count
#     3) mean_topic_share_all_verified_topics
# - top 1 by-group insight per category using the same ranking
#
# Appendix:
# - all other accepted insights

packaging = apply_deterministic_packaging(
    insights_flat_df,
    output_group_key=OUTPUT_GROUP_KEY,
    packaging_cfg=PACKAGING_CFG,
)

accepted_pack_df = packaging["accepted_df"]

print(f"Total insights before packaging: {len(insights_flat_df)}")
print(f"Accepted after thresholds: {len(accepted_pack_df)}")

curated_df, dedup_audit_df = dedupe_packaged_insights(
    accepted_pack_df,
    dedupe_cfg=DEDUPE_CFG,
)

curated_df = curated_df.copy()
curated_df["looker_url"] = curated_df["top_project_ids"].apply(
    lambda ids: build_looker_project_url(
        base_url=LOOKER_BASE_URL,
        project_ids=ids,
        filter_field=LOOKER_FILTER_FIELD,
        fields=LOOKER_FIELDS,
        limit=LOOKER_LIMIT,
        max_ids=LOOKER_ID_LIMIT,
    )
)

# main_min_verification_ratio gates eligibility for main-section placement;
# sourced from params.yaml → analysis.packaging.main_min_verification_ratio.
curated_df = assign_topline_sections_simple(
    curated_df,
    output_group_key=OUTPUT_GROUP_KEY,
    main_cross_limit=PACKAGING_CFG["main_cross_limit"],
    main_min_verification_ratio=MAIN_MIN_VERIFICATION_RATIO,
)

main_cross_df = curated_df[curated_df["report_section"] == "main_cross"].copy()
main_by_group_df = curated_df[curated_df["report_section"] == "main_by_group"].copy()
appendix_df = curated_df[
    curated_df["report_section"].isin(["appendix_cross", "appendix_by_group"])
].copy()

accepted_ids = set(curated_df["insight_id"])
rejected_ids = set(insights_flat_df["insight_id"]) - accepted_ids

# export project_id list by insight
insight_project_df = (
    insight_topic_support_df[
        insight_topic_support_df["insight_id"].isin(accepted_ids)
    ][["insight_id", "group_value", "topic_id"]]
    .merge(
        project_topic_bridge_df[["project_id", GROUPBY_FIELD, "topic_id"]],
        left_on=["group_value", "topic_id"],
        right_on=[GROUPBY_FIELD, "topic_id"],
        how="left",
    )
    [["insight_id", "project_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
insight_project_df.to_csv(OUT("insights", "insight_project_bridge.csv"), index=False)

print(f"Accepted before dedupe: {len(accepted_pack_df)}")
print(f"Dropped as obvious duplicates: {len(dedup_audit_df)}")
print(f"Accepted after dedupe: {len(curated_df)}")
print(f"Main cross-category selected: {len(main_cross_df)}")
print(f"Main by-group selected: {len(main_by_group_df)}")
print(f"Appendix selected: {len(appendix_df)}")
print(f"{len(insight_project_df):,} insight-project pairs across {insight_project_df['insight_id'].nunique()} insights")

Total insights before packaging: 47
Accepted after thresholds: 44
Accepted before dedupe: 44
Dropped as obvious duplicates: 0
Accepted after dedupe: 44
Main cross-category selected: 5
Main by-group selected: 14
Appendix selected: 25
145,428 insight-project pairs across 44 insights


---
## Step 11 — Final Outputs & Manifests

Persists all accepted and rejected outputs, builds the DOCX report, and writes
stage and pipeline manifests. Running this cell again after a partial run is
safe — all writes are deterministic.

In [17]:
# ── FINAL OUTPUTS + REPORTING ──────────────────────────────────────────────
# Persist accepted/rejected outputs, evidence exports, chart data, DOCX, and manifests.

structured = build_structured_from_curated(
    curated_df,
    output_group_key=OUTPUT_GROUP_KEY,
)

write_json(OUT("insights", "insights_structured.json"), structured)

rejected_df = insights_flat_df[
    ~insights_flat_df["insight_id"].isin(curated_df["insight_id"])
].copy()
write_json(
    OUT("insights", "rejected_insights.json"),
    rejected_df.to_dict(orient="records"),
)

insight_topic_support_df[
    insight_topic_support_df["insight_id"].isin(curated_df["insight_id"])
].to_csv(
    OUT("insights", "insight_topic_support.csv"),
    index=False,
)

chart_ready_insights_df = curated_df[
    [
        "run_id",
        "insight_id",
        "title",
        "section",
        "category_bucket",
        "supporting_project_count",
        "verified_topic_count",
        "mean_topic_share_all_verified_topics",
        "verification_ratio",
        "report_section",
        "report_order",
    ]
].rename(columns={"title": "theme"})

chart_ready_group_support_df = insight_topic_support_df[
    insight_topic_support_df["insight_id"].isin(curated_df["insight_id"])
].copy()

chart_ready_insights_df.to_csv(
    OUT("chart_data", "chart_ready_insights.csv"),
    index=False,
)
chart_ready_group_support_df.to_csv(
    OUT("chart_data", "chart_ready_group_support.csv"),
    index=False,
)

docx_path = OUT("reports", "trend_tracker_report.docx")
build_packaged_report_docx(
    structured=structured,
    output_path=docx_path,
    output_group_key=OUTPUT_GROUP_KEY,
    report_cfg=CFG["output"],
    project_count=len(df),
    run_id=RUN_ID,
)

groups_failed = sorted(set(NMF_GROUPS_FAILED) | set(LABELING_FAILED_GROUPS) | set(SYNTHESIS_FAILED_GROUPS))
eligible_groups = list(per_group_results.keys())
manifest_status = "failure" if groups_failed else "success"

stage_manifest_path = OUT("metadata", "stage_manifest_03_insights_generation.json")
finalize_stage_manifest(
    STAGE_MANIFEST,
    output_path=stage_manifest_path,
    status=manifest_status,
    input_artifacts=[
        artifact_meta(ROOT / "OUTPUTS/prepared/05_enriched.parquet", "enriched_parquet"),
        artifact_meta(ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json", "stage_manifest_01"),
        artifact_meta(ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json", "stage_manifest_02"),
    ],
    output_artifacts=[
        artifact_meta(COPIED_CONFIG_PATH, "resolved_params_yaml"),
        artifact_meta(OUT("analysis", "category_tfidf.csv"), "category_tfidf_csv"),
        artifact_meta(OUT("analysis", "nmf_topics.csv"), "nmf_topics_csv"),
        artifact_meta(OUT("analysis", "nmf_weights.csv"), "nmf_weights_csv"),
        artifact_meta(OUT("analysis", "project_topic_bridge.csv"), "project_topic_bridge_csv"),
        artifact_meta(OUT("analysis", "llm_topic_labels.json"), "llm_topic_labels_json"),
        artifact_meta(OUT("insights", "insights_candidates.json"), "insights_candidates_json"),
        artifact_meta(OUT("insights", "insights_structured.json"), "insights_structured_json"),
        artifact_meta(OUT("insights", "rejected_insights.json"), "rejected_insights_json"),
        artifact_meta(OUT("insights", "insights_flat.csv"), "insights_flat_csv"),
        artifact_meta(OUT("insights", "insight_topic_support.csv"), "insight_topic_support_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_insights.csv"), "chart_ready_insights_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_group_support.csv"), "chart_ready_group_support_csv"),
        artifact_meta(OUT("reports", "trend_tracker_report.docx"), "trend_tracker_report_docx"),
    ],
    row_counts={
        "input_projects": int(len(raw_df)),
        "filtered_projects": int(len(df)),
        "eligible_groups": int(len(eligible_groups)),
        "groups_skipped": int(len(NMF_GROUPS_SKIPPED)),
        "groups_failed": int(len(groups_failed)),
        "topics_generated": int(len(topics_df)),
        "insights_generated": int(len(insights_flat_df)),
        "insights_accepted": int(len(accepted_ids)),
        "insights_rejected": int(len(rejected_ids)),
    },
    key_params={
        "group_by": GROUPBY_FIELD,
        "filter_fields_key": FILTER_FIELDS_KEY,
        "min_group_projects": CFG["analysis"]["min_group_projects"],
        "topic_assignment_threshold": CFG["analysis"]["topic_assignment_threshold"],
        "model_labeling": MODEL_LABELING,
        "model_synthesis": MODEL_SYNTHESIS,
        "verification_config": VERIFY_CFG,
        "packaging_config": PACKAGING_CFG,
        "dedupe_config": DEDUPE_CFG,
    },
    warnings_path=WARNINGS_PATH,
)

pipeline_manifest_path = OUT("metadata", "pipeline_manifest.json")
build_pipeline_manifest(
    output_path=pipeline_manifest_path,
    run_id=RUN_ID,
    run_date=RUN_DATE,
    status=manifest_status,
    config_path=MANIFEST_CFG_PATH,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
    filter_spec_path=FILTER_SPEC_PATH,
    filter_summary_path=FILTER_SUMMARY_PATH,
    stage_manifest_paths=[
        ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json",
        ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json",
        stage_manifest_path,
    ],
    warnings_01_path=ROOT / "OUTPUTS/prepared/warnings_01.jsonl",
    warnings_02_path=ROOT / "OUTPUTS/enrichment/warnings_02.jsonl",
    warnings_03_path=WARNINGS_PATH,
    final_outputs={
        "insights_structured_json": str(OUT("insights", "insights_structured.json")),
        "rejected_insights_json": str(OUT("insights", "rejected_insights.json")),
        "insights_flat_csv": str(OUT("insights", "insights_flat.csv")),
        "insight_topic_support_csv": str(OUT("insights", "insight_topic_support.csv")),
        "chart_ready_insights_csv": str(OUT("chart_data", "chart_ready_insights.csv")),
        "chart_ready_group_support_csv": str(OUT("chart_data", "chart_ready_group_support.csv")),
        "trend_tracker_report_docx": str(OUT("reports", "trend_tracker_report.docx")),
    },
)

print(f"Accepted insights: {len(accepted_ids):,}")
print(f"Rejected insights: {len(rejected_ids):,}")
print(f"Docx saved → {docx_path}")
print(f"Stage manifest written → {stage_manifest_path}")
print(f"Pipeline manifest written → {pipeline_manifest_path}")

Accepted insights: 44
Rejected insights: 3
Docx saved → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/project_category/2026-04-10/20260410_111800_project_category_77e319db/reports/trend_tracker_report.docx
Stage manifest written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/project_category/2026-04-10/20260410_111800_project_category_77e319db/metadata/stage_manifest_03_insights_generation.json
Pipeline manifest written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/project_category/2026-04-10/20260410_111800_project_category_77e319db/metadata/pipeline_manifest.json


In [71]:
# --- Charting UDFs ---

def counts_pct(s):
    """Value counts with pct for a series."""
    c = s.value_counts(dropna=False).rename_axis('category').reset_index(name='count')
    c['pct'] = (c['count'] / c['count'].sum()).round(3)
    return c

def quintile_bins(df, col, label, bins=None):
    s = df[col].dropna()
    if bins is None:
        cut, bins = pd.qcut(s, 5, retbins=True, duplicates='drop')
    else:
        cut = pd.cut(s, bins=bins, include_lowest=True)
    result = cut.value_counts().sort_index().reset_index()
    result.columns = [label, 'count']
    result['pct'] = (result['count'] / result['count'].sum()).round(3)
    result['min'] = [iv.left for iv in result[label].cat.categories]
    result['max'] = [iv.right for iv in result[label].cat.categories]
    return result[['min', 'max', 'count', 'pct']], bins

def fy_from_date(s):
    """Return fiscal year string (Jul start) from a date series."""
    s = pd.to_datetime(s)
    return s.apply(lambda d: f'FY{(d.year + 1) % 100:02d}' if d.month >= 7 else f'FY{d.year % 100:02d}')

def fy_half(s):
    """Return H1 (Jul-Dec) or H2 (Jan-Jun) from a date series."""
    return pd.to_datetime(s).dt.month.apply(lambda m: 'H1 (Jul-Dec)' if m >= 7 else 'H2 (Jan-Jun)')

def assign_efs(row):
    rural = row['school_is_underserved_rural'] == 'Yes'
    race  = row['school_is_historically_underrepresented_race'] == 'Yes'
    inc   = row['school_is_low_income'] == 'Yes'
    if race and inc:  return 'Race+Inc'
    if rural:         return 'Rural'
    if race:          return 'Race'
    if inc:           return 'Inc'
    return 'NonEFS'

# --- Chart params ---

EFS_ORDER = ['Race+Inc', 'Rural', 'Race', 'Inc', 'NonEFS']
RACE_COLS = {'Black': 'school_percent_black_imputed', 'Latinx': 'school_percent_latinx_imputed',
             'Asian': 'school_percent_asian_imputed', 'White': 'school_percent_white_imputed'}

# Pre-join once: attach all cluster fields to insight-project bridge
df_full = df.copy()
df_full['efs_category'] = df_full.apply(assign_efs, axis=1)
df_full['_fy']          = fy_from_date(df_full['posted_date'])
df_full['_half']        = fy_half(df_full['posted_date'])
df_full['_exp_years']   = pd.to_datetime(df_full['posted_date']).dt.year - df_full['teacher_start_teaching_year']

df_enriched = (
    insight_project_df
    .merge(df, on='project_id', how='left')
)
required_cols = [
    'project_id', 'funded_date', 'total_cost', 'posted_date',
    'teacher_start_teaching_year', 'metro_type_at_time_of_posting',
    'grade_band', 'school_is_low_income', 'school_is_historically_underrepresented_race',
    'school_is_underserved_rural', 'school_enrollment',
    'school_percent_black_imputed', 'school_percent_latinx_imputed',
    'school_percent_asian_imputed', 'school_percent_white_imputed',
    'active_geo_region',
]
missing = [c for c in required_cols if c not in df.columns]
print(missing)
df_enriched['efs_category'] = df_enriched.apply(assign_efs, axis=1)
df_enriched['_fy']        = fy_from_date(df_enriched['posted_date'])
df_enriched['_half']      = fy_half(df_enriched['posted_date'])
df_enriched['_exp_years'] = pd.to_datetime(df_enriched['posted_date']).dt.year - df_enriched['teacher_start_teaching_year']

insight_charts = {}
cost_bins = None
exp_bins  = None

for insight_id, df_i in [('full_corpus', df_full), ('total_sample', df_enriched)] + list(df_enriched.groupby('insight_id')):
    
    enroll = df_i['school_enrollment']

    # EFS ordering
    efs = counts_pct(df_i['efs_category'])
    efs['category'] = pd.Categorical(efs['category'], categories=EFS_ORDER, ordered=True)
    efs = efs.sort_values('category').reset_index(drop=True)

    # Posting distribution (normalized so it can be compared/charted like other fields)
    posting = (
        df_i.assign(
            half_short=df_i['_half'].str.extract(r'^(H[12])', expand=False),
            fy_num=df_i['_fy'].str.replace('FY', '', regex=False).astype(int),
        )
        .groupby(['_fy', 'half_short', 'fy_num'], dropna=False)
        .size()
        .reset_index(name='count')
        .rename(columns={'_fy': 'fy'})
    )
    posting['category'] = posting['fy'] + ' ' + posting['half_short']
    posting['period_sort'] = posting['fy_num'] * 10 + posting['half_short'].map({'H1': 1, 'H2': 2})
    posting['pct'] = (posting['count'] / posting['count'].sum()).round(3)
    posting = posting[['category', 'count', 'pct', 'period_sort']]

    # Use the first df_i value to lock quintile bin thresholds for all future df_i (e.g. full_corpus)
    cost_result, cost_bins = quintile_bins(df_i.dropna(subset=['total_cost']), 'total_cost', 'cost_bucket', bins=cost_bins)
    exp_result,  exp_bins  = quintile_bins(df_i.dropna(subset=['_exp_years']), '_exp_years', 'experience_bucket', bins=exp_bins)
    
    insight_charts[insight_id] = {
        'project_count': len(df_i),
        'cost_distr':    cost_result,
        'posting':       posting,
        'exper_distr':   exper_result,
        'metro':         counts_pct(df_i['metro_type_at_time_of_posting']),
        'grade':         counts_pct(df_i['grade_band']),
        'efs':           efs,
        'funding':       counts_pct(pd.Series(np.select(
                             [df_i['funded_date'].notna(), pd.to_datetime(df_i['expiration_date']) <= pd.Timestamp.today()],
                             ['Funded', 'Expired'], default='Live'
                         ), index=df_i.index)),
        'race':          pd.DataFrame([
                             {'race': r, 'weighted_avg_pct': round((df_i[col] * enroll).sum() / enroll.sum(), 3)}
                             for r, col in RACE_COLS.items()
                         ]),
        #'region':        counts_pct(df_i['active_geo_region']),
    }

print(f"Built chart data for {len(insight_charts)} insights")

# Access to aan insight's charts
iid = 'KI_008'

for chart_name, chart_data in insight_charts[iid].items():
    print(f"\n--- {chart_name} ---")
    if isinstance(chart_data, pd.DataFrame):
        print(chart_data.to_string(index=False))
    else:
        print(chart_data)

['active_geo_region']
Built chart data for 46 insights

--- project_count ---
5829

--- cost_distr ---
    min        max  count   pct
117.659    237.950   1108 0.190
237.950    370.550   1477 0.253
370.550    536.454   1334 0.229
536.454    750.730   1110 0.190
750.730 121779.980    800 0.137

--- posting ---
category  count   pct  period_sort
 FY23 H2    147 0.025          232
 FY24 H1   1671 0.287          241
 FY24 H2    860 0.148          242
 FY25 H1   1130 0.194          251
 FY25 H2    796 0.137          252
 FY26 H1   1034 0.177          261
 FY26 H2    191 0.033          262

--- exper_distr ---
   min  max  count   pct
-3.001  4.0    828 0.194
 4.000  9.0    813 0.191
 9.000 16.0    901 0.211
16.000 23.0    924 0.217
23.000 59.0    795 0.187

--- metro ---
    category  count   pct
       Urban   3230 0.554
    Suburban   1456 0.250
       Rural    558 0.096
        Town    338 0.058
Unclassified    247 0.042

--- grade ---
     category  count   pct
Grades PreK-2   2139 0.3

In [72]:
def tvd(df_a, df_b, key_col, val_col='pct'):
    """Total variation distance between two categorical distributions."""
    a = df_a[[key_col, val_col]].copy()
    b = df_b[[key_col, val_col]].copy()
    a[key_col] = a[key_col].astype(str)
    b[key_col] = b[key_col].astype(str)
    merged = a.merge(b, on=key_col, how='outer', suffixes=('_a', '_b')).fillna(0)
    return round((merged[f'{val_col}_a'] - merged[f'{val_col}_b']).abs().sum() / 2, 4)

def race_dist(df_a, df_b):
    """Sum of absolute differences in weighted_avg_pct across races, divided by 100 to normalize to 0-1 scale (matching TVD)."""
    merged = df_a.merge(df_b, on='race', suffixes=('_a', '_b'))
    return round((merged['weighted_avg_pct_a'] - merged['weighted_avg_pct_b']).abs().sum() / 100, 4)
    
corpus        = insight_charts['full_corpus']
total_n       = insight_charts['total_sample']['project_count']

distinctness = []

for insight_id, charts in insight_charts.items():
    if insight_id in ('total_sample', 'full_corpus'):
        continue
    distinctness.append({
        'insight_id':    insight_id,
        'project_count': charts['project_count'],
        'pct_of_total':  round(charts['project_count'] / total_n, 4),
        'funding':       tvd(charts['funding'],   corpus['funding'],   'category'),
        'cost_distr':    tvd(charts['cost_distr'], corpus['cost_distr'], 'min'),
        'exper_distr':   tvd(charts['exper_distr'],  corpus['exper_distr'],  'min'),
        'metro':         tvd(charts['metro'],     corpus['metro'],     'category'),
        'grade':         tvd(charts['grade'],     corpus['grade'],     'category'),
        'efs':           tvd(charts['efs'],        corpus['efs'],      'category'),
        'race':          race_dist(charts['race'], corpus['race']),
        'posting':       tvd(charts['posting'],    corpus['posting'],    'category'),
        #'region':        tvd(charts['region'],    corpus['region'],    'category'),
    })

# Add a summary column: mean TVD across all chart types (excluding project_count cols)
distinctness_df = pd.DataFrame(distinctness)
distinctness_df = distinctness_df[distinctness_df['project_count'] > 500].reset_index(drop=True)
tvd_cols = ['funding', 'cost_distr', 'exper_distr', 'metro', 'grade', 'efs', 'race']
distinctness_df['mean_tvd'] = distinctness_df[tvd_cols].mean(axis=1).round(4)
distinctness_df = distinctness_df.sort_values('mean_tvd', ascending=False).reset_index(drop=True)
distinctness_df

,insight_id,project_count,pct_of_total,funding,cost_distr,exper_distr,metro,grade,efs,race,posting,mean_tvd
0,BG_014,2557,0.0176,0.0,0.0830,0.0,0.1000,0.1960,0.1465,0.2190,0.1405,0.1064
1,BG_026,868,0.0060,0.0,0.2120,0.0,0.0465,0.3510,0.0555,0.0551,0.0710,0.1029
2,KI_006,8104,0.0557,0.0,0.2205,0.0,0.1050,0.0330,0.1495,0.1858,0.0965,0.0991
3,BG_017,8273,0.0569,0.0,0.1840,0.0,0.0890,0.0920,0.1430,0.1739,0.0405,0.0974
4,KI_005,2784,0.0191,0.0,0.0735,0.0,0.0895,0.1340,0.1405,0.2035,0.1295,0.0916
5,BG_019,8310,0.0571,0.0,0.1825,0.0,0.0660,0.1285,0.1235,0.1371,0.0910,0.0911
6,BG_018,10229,0.0703,0.0,0.1680,0.0,0.0705,0.1265,0.1075,0.1329,0.0405,0.0865
7,BG_031,1689,0.0116,0.0,0.1860,0.0,0.0835,0.0855,0.1160,0.1330,0.0610,0.0863
8,BG_027,588,0.0040,0.0,0.1705,0.0,0.0525,0.3035,0.0345,0.0255,0.0545,0.0838
9,BG_010,1072,0.0074,0.0,0.2875,0.0,0.0410,0.0975,0.0665,0.0695,0.1080,0.0803


In [73]:
from IPython.display import display, HTML

tvd_cols = ['funding', 'cost_distr', 'exper_distr', 'metro', 'grade', 'efs', 'race', 'posting',] #'region'

sort_col = {
    'funding': 'category',
    'posting': 'period_sort',
    'metro': 'category',
    'grade': 'category',
    'efs': 'category',
    'region': 'category',
    'cost_distr': 'min',
    'exper_distr': 'min',
    'race': 'race',
}

for field in tvd_cols:
    top10 = distinctness_df.nlargest(10, field)['insight_id'].tolist()
    ids   = ['full_corpus'] + top10

    tables_html = []
    for iid in ids:
        df_t = insight_charts[iid][field].sort_values(sort_col[field]).reset_index(drop=True)
        label = f"<b>{iid}</b><br><small>tvd=0</small>" if iid == 'full_corpus' else f"{iid}<br><small>tvd={distinctness_df.loc[distinctness_df['insight_id']==iid, field].values[0]}</small>"
        tbl = df_t.to_html(index=False, border=1)
        tables_html.append(f"<div style='display:inline-block; vertical-align:top; margin-right:20px'>{label}{tbl}</div>")

    display(HTML(
        f"<h3 style='margin-top:30px'>{field}</h3>"
        + "".join(tables_html)
    ))

category,count,pct
Funded,420625,1.0
category,count,pct
Funded,2557,1.0
category,count,pct
Funded,868,1.0
category,count,pct
Funded,8104,1.0
category,count,pct
Funded,8273,1.0
category,count,pct


min,max,count,pct
117.659,237.950,84132,0.2
237.950,370.550,84122,0.2
370.550,536.454,84121,0.2
536.454,750.730,84126,0.2
750.730,121779.980,84124,0.2
min,max,count,pct
117.659,237.950,42,0.065
237.950,370.550,29,0.045
370.550,536.454,159,0.246
536.454,750.730,165,0.255


min,max,count,pct
-3.001,4.0,828,0.194
4.000,9.0,813,0.191
9.000,16.0,901,0.211
16.000,23.0,924,0.217
23.000,59.0,795,0.187
min,max,count,pct
-3.001,4.0,828,0.194
4.000,9.0,813,0.191
9.000,16.0,901,0.211
16.000,23.0,924,0.217


category,count,pct
Rural,44733,0.106
Suburban,118845,0.283
Town,27904,0.066
Unclassified,21842,0.052
Urban,207301,0.493
category,count,pct
Rural,626,0.077
Suburban,1852,0.229
Town,387,0.048
Unclassified,392,0.048


category,count,pct
Grades 3-5,120862,0.287
Grades 6-8,74787,0.178
Grades 9-12,74451,0.177
Grades PreK-2,150525,0.358
category,count,pct
Grades 3-5,211,0.243
Grades 6-8,344,0.396
Grades 9-12,269,0.310
Grades PreK-2,44,0.051
category,count,pct


category,count,pct
Race+Inc,284919,0.677
Rural,26040,0.062
Race,20681,0.049
Inc,37276,0.089
NonEFS,51709,0.123
category,count,pct
Race+Inc,6697,0.826
Rural,294,0.036
Race,251,0.031
Inc,544,0.067


race,weighted_avg_pct
Asian,6.205
Black,17.937
Latinx,40.169
White,25.407
race,weighted_avg_pct
Asian,5.455
Black,14.233
Latinx,33.503
White,36.189
race,weighted_avg_pct


category,count,pct,period_sort
FY23 H1,2,0.000,231
FY23 H2,10619,0.025,232
FY24 H1,105260,0.250,241
FY24 H2,58202,0.138,242
FY25 H1,84215,0.200,251
FY25 H2,64193,0.153,252
FY26 H1,78376,0.186,261
FY26 H2,19758,0.047,262
category,count,pct,period_sort
FY23 H2,28,0.018,232


In [74]:
import json
from IPython.display import display, HTML

# ── config ──────────────────────────────────────────────────────────────────
PALETTE = ['#6b6b6b', '#1a6fb5', '#0d8c5e', '#c94c1f', '#a83060', '#9a6200',
           '#5b3fa6', '#1a8fa6', '#6b8c1a', '#a65c1a', '#1a5ca6']

SORT_KEY = {
    'funding': 'category',
    'posting': 'period_sort',
    'metro': 'category',
    'grade': 'category',
    'efs': 'category',
    'region': 'category',
    'race': 'race',
    'cost_distr': 'min',
    'exper_distr': 'min',
}

FIELD_LABELS = {
    'funding': 'Funding Status',
    'posting': 'Posting Period',
    'metro': 'Metro Type',
    'grade': 'Grade Level',
    'efs': 'EFS Classification',
    'region': 'Geographic Region',
    'race': 'Student Race',
    'cost_distr': 'Project Cost',
    'exper_distr': 'Teacher Experience',
}

def extract(field, iid):
    df_t = insight_charts[iid][field].sort_values(SORT_KEY[field]).reset_index(drop=True)
    val_col = 'weighted_avg_pct' if field == 'race' else 'pct'
    if field == 'cost_distr':
        labels = [f"${r['min']:,.0f}–${r['max']:,.0f}" for _, r in df_t.iterrows()]
    elif field == 'exper_distr':
        labels = [f"{int(r['min'])}–{int(r['max'])}yr" for _, r in df_t.iterrows()]
    else:
        lbl_col = 'race' if field == 'race' else 'category'
        labels = df_t[lbl_col].astype(str).tolist()
    values = [round(float(v), 5) if not pd.isna(v) else 0.0 for v in df_t[val_col]]
    return labels, values

PLOT_FIELDS = [f for f in tvd_cols if f in SORT_KEY]

title_lookup = curated_df.set_index('insight_id')['title'].to_dict()
title_lookup['full_corpus'] = 'Full Corpus (baseline)'

# ── build html ───────────────────────────────────────────────────────────────
divs, all_cfgs = [], []

for field in PLOT_FIELDS:
    top10 = distinctness_df.nlargest(10, field)['insight_id'].tolist()
    ids   = ['full_corpus'] + top10

    fc_labels, fc_vals = extract(field, 'full_corpus')
    fc_lookup = dict(zip(fc_labels, fc_vals))
    all_vals = []
    for iid in ids:
        _, v = extract(field, iid)
        all_vals.extend(v)
    xmax    = round(max(all_vals) * 1.18, 4)
    is_pct  = field != 'race'

    cards = []
    for rank, iid in enumerate(ids):
        labels, vals = extract(field, iid)
        color = PALETTE[rank]
        title = title_lookup.get(iid, iid)
        h = max(140, len(labels) * 28 + 52)
        cid = f'c{len(all_cfgs)}'

        if iid == 'full_corpus':
            badge = '— baseline'
            color_badge = '#aaa'
        else:
            tvd_val = distinctness_df.loc[distinctness_df['insight_id']==iid, field].values[0]
            badge = f'tvd {tvd_val:.3f}'
            color_badge = color

        n_str = f"{insight_charts[iid]['project_count']:,}"

        cards.append(
            f'<div style="flex:0 0 178px;background:#fff;border:0.5px solid #d4d0c8;border-top:2.5px solid {color};border-radius:0 0 5px 5px;padding:12px 11px 8px">'
            f'<div style="font-size:10px;font-weight:700;letter-spacing:.01em;color:#111;font-family:\'Helvetica Neue\',sans-serif;line-height:1.35;text-transform:none;margin-bottom:1px" title="{title}">{title}</div>'
            f'<div style="font-size:9px;margin:3px 0 9px;font-family:\'Helvetica Neue\',sans-serif">'
            f'<span style="color:{color_badge};font-weight:600">n={n_str}</span>'
            f'<span style="color:#ccc"> · </span>'
            f'<span style="color:#aaa;font-weight:400">{badge}</span>'
            f'</div>'
            f'<div style="position:relative;height:{h}px"><canvas id="{cid}" role="img" aria-label="{FIELD_LABELS[field]} chart for {iid}"></canvas></div>'
            f'</div>'
        )

        all_cfgs.append({
            'id': cid, 'labels': labels, 'data': vals,
            'color': color, 'xmax': xmax, 'pct': is_pct,
            'baseline': [fc_lookup.get(lbl, None) for lbl in labels]
        })

    divs.append(
        f'<div style="margin:3rem 0 0">'
        f'<div style="display:flex;align-items:center;gap:0;margin-bottom:.9rem">'
        f'<div style="width:3px;height:22px;background:#111;margin-right:10px;flex-shrink:0"></div>'
        f'<span style="font-size:14px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#111;font-family:\'Helvetica Neue\',sans-serif">{FIELD_LABELS[field]}</span>'
        f'</div>'
        f'<div style="display:flex;gap:8px;overflow-x:auto;padding-bottom:6px">{"".join(cards)}</div>'
        f'</div>'
    )

# ── assemble ─────────────────────────────────────────────────────────────────
cfg_json = json.dumps(all_cfgs)

html = f'''
<div style="background:#f7f5f0;padding:2rem;font-family:\'Helvetica Neue\',sans-serif">
  <div style="margin-bottom:2rem;padding-bottom:1.2rem;border-bottom:2px solid #111">
    <div style="font-size:10px;font-weight:700;letter-spacing:.15em;text-transform:uppercase;color:#999;margin-bottom:4px">Cluster Analysis</div>
    <div style="font-size:20px;font-weight:700;color:#111;letter-spacing:-.02em">Distribution Profiles by Insight</div>
    <div style="font-size:11px;color:#888;margin-top:4px">Baseline (full corpus) vs. top 5 most distinct insights per field · TVD = total variation distance from baseline</div>
  </div>
  {"".join(divs)}
  <div style="margin-top:2.5rem;padding-top:1rem;border-top:0.5px solid #ccc;font-size:9px;color:#aaa;letter-spacing:.05em;text-transform:uppercase">Generated from insight_charts · {len(distinctness_df)} insights shown</div>
</div>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<script>
(function poll() {{
  if (!window.Chart) {{ setTimeout(poll, 60); return; }}
  var cfgs = {cfg_json};
  cfgs.forEach(function(c) {{
    var el = document.getElementById(c.id);
    if (!el) return;
    new Chart(el, {{
      type: 'bar',
      plugins: [{{
        id: 'bl',
        afterDatasetsDraw: function(chart) {{
          var bl = chart.options.plugins.bl;
          if (!bl) return;
          var ctx = chart.ctx, xs = chart.scales.x, ys = chart.scales.y;
          bl.forEach(function(val, i) {{
            if (val === null) return;
            var x = xs.getPixelForValue(val);
            var y = ys.getPixelForValue(i);
            var h = ys.height / (ys.ticks.length * 2.2);
            ctx.save();
            ctx.beginPath();
            ctx.moveTo(x, y - h); ctx.lineTo(x, y + h);
            ctx.strokeStyle = 'rgba(0,0,0,0.65)';
            ctx.lineWidth = 1.5;
            ctx.stroke();
            ctx.restore();
          }});
        }}
      }}],
      data: {{
        labels: c.labels,
        datasets: [{{
          data: c.data,
          backgroundColor: c.color + '99',
          borderColor:     c.color,
          borderWidth: 1,
          borderRadius: 2,
        }}]
      }},
      options: {{
        indexAxis: 'y',
        responsive: true,
        maintainAspectRatio: false,
        plugins: {{
          bl: c.baseline,
          legend: {{ display: false }},
          tooltip: {{
            callbacks: {{
              label: function(ctx) {{
                return c.pct
                  ? ' ' + (ctx.raw * 100).toFixed(1) + '%'
                  : ' ' + ctx.raw.toFixed(1) + '%';
              }}
            }}
          }}
        }},
        scales: {{
          x: {{
            max: c.xmax,
            grid: {{ color: '#ebe8e0', lineWidth: 0.5 }},
            ticks: {{
              font: {{ size: 9, family: "'Helvetica Neue', sans-serif" }},
              color: '#bbb',
              maxTicksLimit: 4,
              callback: function(v) {{
                return c.pct ? (v * 100).toFixed(0) + '%' : v.toFixed(0) + '%';
              }}
            }},
            border: {{ display: false }}
          }},
          y: {{
            grid: {{ display: false }},
            ticks: {{
              font: {{ size: 9, family: "'Helvetica Neue', sans-serif" }},
              color: '#444',
              crossAlign: 'far',
            }},
            border: {{ display: false }}
          }}
        }}
      }}
    }});
  }});
}})();
</script>
'''

display(HTML(html))

In [83]:
def _find_first(obj):
    if isinstance(obj, dict) and 'insight_id' in obj: return obj
    if isinstance(obj, list):
        for i in obj:
            r = _find_first(i)
            if r: return r
    if isinstance(obj, dict):
        for v in obj.values():
            r = _find_first(v)
            if r: return r
_s = _find_first(structured)
if _s: print('Structured keys:', list(_s.keys()))
print('insights_flat_df columns:', list(insights_flat_df.columns))

Structured keys: ['insight_id', 'title', 'what_seeing', 'why_easy_to_miss', 'source_topics', 'supporting_project_count', 'verified_topic_count', 'verification_ratio', 'mean_topic_share_all_verified_topics', 'top_project_ids', 'looker_url', 'report_section', 'report_order', 'warnings']
insights_flat_df columns: ['run_id', 'insight_id', 'section', 'category_bucket', 'title', 'what_seeing', 'why_easy_to_miss', 'source_topics_verified', 'source_topics_claimed', 'claimed_topic_count', 'verified_topic_count', 'verification_ratio', 'supporting_project_count', 'mean_topic_share_all_verified_topics', 'top_project_ids']


In [95]:
"""
HTML Report Builder v2 — add to notebook after Step 11
Prereqs: report_template.html and chart.umd.min.js in ROOT
"""

import json, math, base64

# generates a favicon from an emoji
emoji = '✏️'   # change to whatever you want: 📊 🎓 📚 ✏️ etc.
svg = f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 100 100"><text y=".9em" font-size="90">{emoji}</text></svg>'
favicon_uri = 'data:image/svg+xml;base64,' + base64.b64encode(svg.encode()).decode()

# ── config ────────────────────────────────────────────────────────────────────
DETAIL_FIELDS = ['metro', 'grade', 'efs', 'race']
FIELD_META = {
    'metro': {'val_col': 'pct',              'sort_col': 'category'},
    'grade': {'val_col': 'pct',              'sort_col': 'category'},
    'efs':   {'val_col': 'pct',              'sort_col': 'category'},
    'race':  {'val_col': 'weighted_avg_pct', 'sort_col': 'race'},
}
EFS_ORDER = ['Race+Inc', 'Rural', 'Race', 'Inc', 'NonEFS']

# Human-readable suffixes for separator badges
SEP_SUFFIX = {
    'metro': 'school coverage',
    'grade': 'coverage',
    'efs':   'school coverage',
    'race':  'student population',
}

def _safe_float(v):
    try:
        f = float(v)
        return 0.0 if math.isnan(f) or math.isinf(f) else f
    except (TypeError, ValueError):
        return 0.0


def df_to_chart(df, field):
    """Convert chart df to (labels, values) lists."""
    meta     = FIELD_META[field]
    val_col  = meta['val_col']
    sort_col = meta['sort_col']
    df_s = df.copy()
    df_s[sort_col] = df_s[sort_col].astype(str)
    if field == 'efs':
        order_map = {v: i for i, v in enumerate(EFS_ORDER)}
        df_s['_ord'] = df_s['category'].map(order_map).fillna(99)
        df_s = df_s.sort_values('_ord').reset_index(drop=True)
    else:
        df_s = df_s.sort_values(sort_col).reset_index(drop=True)
    labels = df_s[sort_col].tolist()
    values = [_safe_float(v) for v in df_s[val_col]]
    return labels, values


def compute_separators(ins_charts, fc_charts_local):
    """
    Find top-2 field+value combinations where this insight is most
    above-average vs full population. Returns dict with sep1/sep2 keys.
    Race values are on 0-100 scale; all others on 0-1. Normalise to ppt.
    """
    candidates = []
    for field in DETAIL_FIELDS:
        ch = ins_charts.get(field)
        fc = fc_charts_local.get(field)
        if not ch or not fc:
            continue
        is_race = field == 'race'
        scale   = 1 if is_race else 100        # convert both to ppt
        fc_map  = dict(zip(fc['labels'], fc['values']))
        for label, val in zip(ch['labels'], ch['values']):
            corpus_val = fc_map.get(label, 0)
            diff_ppt   = (val - corpus_val) * (1 if is_race else 100)
            if diff_ppt > 0.5:                 # ignore noise < 0.5 ppt
                candidates.append({
                    'field':     field,
                    'label':     label,
                    'diff_ppt':  diff_ppt,
                    'diff_disp': round(diff_ppt, 1),
                    'suffix':    SEP_SUFFIX.get(field, ''),
                })
    candidates.sort(key=lambda x: x['diff_ppt'], reverse=True)
    result = {}
    for n, key in enumerate(['sep1', 'sep2'], 1):
        if len(candidates) >= n:
            c = candidates[n - 1]
            result[f'{key}_val'] = round(c['diff_disp'], 0)
            result[f'{key}_lbl'] = f"More {c['label']} {c['suffix']}".strip()
        else:
            result[f'{key}_val'] = None
            result[f'{key}_lbl'] = None
    return result


# ── full corpus charts + global xmax ────────────────────────────────────────
fc_charts_local = {}
field_xmax      = {}

for field in DETAIL_FIELDS:
    if 'full_corpus' not in insight_charts:
        continue
    fc_df = insight_charts['full_corpus'].get(field)
    if fc_df is None:
        continue
    labels, values = df_to_chart(fc_df, field)
    fc_charts_local[field] = {'labels': labels, 'values': values}
    field_xmax[field] = max(values) * 1.2 if values else 1.0

for iid, charts in insight_charts.items():
    if iid in ('full_corpus', 'total_sample'):
        continue
    for field in DETAIL_FIELDS:
        ch = charts.get(field)
        if ch is None:
            continue
        _, values = df_to_chart(ch, field)
        candidate = max(values) * 1.2 if values else 0.0
        if candidate > field_xmax.get(field, 0):
            field_xmax[field] = candidate

for field in field_xmax:
    field_xmax[field]                         = round(field_xmax[field], 4)
    fc_charts_local.setdefault(field, {})['xmax'] = field_xmax[field]


# ── extract body text from structured dict ───────────────────────────────────
def _extract_body(d):
    what = (d.get('what_seeing') or d.get('what_we_see') or d.get('what')
            or d.get('finding')  or d.get('body') or d.get('description') or '')
    why  = (d.get('why_easy_to_miss') or d.get('why')  or d.get('implication')
            or d.get('body2')          or d.get('context') or '')
    return str(what).strip(), str(why).strip()

insight_bodies = {}

def _walk(obj):
    if isinstance(obj, dict):
        if 'insight_id' in obj:
            iid = obj['insight_id']
            if iid not in insight_bodies:
                what, why = _extract_body(obj)
                insight_bodies[iid] = {'what_we_see': what, 'why_easy_to_miss': why}
        for v in obj.values():
            _walk(v)
    elif isinstance(obj, list):
        for item in obj:
            _walk(item)

_walk(structured)


# ── TVD lookup ────────────────────────────────────────────────────────────────
tvd_lookup = {row['insight_id']: row.to_dict() for _, row in distinctness_df.iterrows()}


# ── filter description from CFG (already in memory) ─────────────────────────
def _fmt_filter(f):
    field = f.get('field', '?')
    op    = f.get('op', '?')
    if op == 'eq':
        return f"{field} = {f.get('value','?')}"
    elif op == 'in':
        vals = f.get('values', f.get('value', '?'))
        if isinstance(vals, list):
            vals = ', '.join(str(v) for v in vals[:5]) + (f' (+{len(vals)-5} more)' if len(vals) > 5 else '')
        return f"{field} in [{vals}]"
    elif op == 'range':
        min_val = f.get('min', '?')
        max_val = f.get('max', None)
        if max_val is None or str(max_val).strip().lower() in ('', 'null', 'none', '?'):
            max_val = 'present'
        return f"{field} from {min_val} to {max_val}"
    elif op == 'not_null':
        return f"{field} is present"
    elif op == 'is_null':
        return f"{field} is absent"
    return f"{field} {op}"

try:
    _filters_list  = CFG.get('analysis', {}).get('filters', [])
    _filter_logic  = CFG.get('analysis', {}).get('filter_logic', 'and').upper()
    _groupby_label = GROUPBY_FIELD.replace('_', ' ')

    if _filters_list:
        _filter_parts = [_fmt_filter(f) for f in _filters_list]
        _filter_str   = f' {_filter_logic} '.join(_filter_parts)
        filter_headline = (f"Insights were found by deep diving on {_groupby_label}, "
                           f"filtered on {_filter_str}")
        filter_detail   = (f"{len(_filters_list)} filter{'s' if len(_filters_list) != 1 else ''} applied "
                           f"({_filter_logic}) · {len(curated_df)} insights accepted from {len(df):,} projects")
    else:
        filter_headline = f"Insights were found by deep diving on {_groupby_label} across {len(df):,} projects"
        filter_detail   = f"No filters applied · {len(curated_df)} insights accepted"
except Exception as e:
    filter_headline = f"Insights were found by deep diving on {GROUPBY_FIELD.replace('_',' ')}"
    filter_detail   = f"No filters applied · {len(curated_df)} insights accepted"
    print(f"Warning: could not read filter config ({e})")


# ── build insights list ───────────────────────────────────────────────────────
def _is_cross(row):
    rs = str(row.get('report_section', '')).lower()
    s  = str(row.get('section', '')).lower()
    return 'cross' in rs or s == 'key_insights' or not row.get('category_bucket') or \
           str(row.get('category_bucket','')).lower() in ('none','')

insights_out = []

for _, row in curated_df.sort_values('supporting_project_count', ascending=False).iterrows():
    iid     = row['insight_id']
    body    = insight_bodies.get(iid, {'what_we_see': '', 'why_easy_to_miss': ''})
    tvd_r   = tvd_lookup.get(iid, {})
    is_cross = _is_cross(row)

    charts_out = {}
    for field in DETAIL_FIELDS:
        ic = insight_charts.get(iid, {}).get(field)
        if ic is None:
            continue
        labels, values = df_to_chart(ic, field)
        fc_labels = fc_charts_local.get(field, {}).get('labels', labels)
        fc_vals   = fc_charts_local.get(field, {}).get('values', [0.0] * len(labels))
        fc_map    = dict(zip(fc_labels, fc_vals))
        baseline  = [fc_map.get(lbl) for lbl in labels]
        charts_out[field] = {
            'labels':   labels,
            'values':   values,
            'baseline': baseline,
            'xmax':     field_xmax.get(field, 1.0),
        }

    seps = compute_separators(charts_out, fc_charts_local)

    insights_out.append({
        'id':                  iid,
        'title':               str(row.get('title', row.get('theme', iid))),
        'what_we_see':         body['what_we_see'],
        'why_easy_to_miss':    body['why_easy_to_miss'],
        'category_bucket':     str(row.get('category_bucket', '')),
        'is_cross':            is_cross,
        'section':             str(row.get('section', '')),
        'report_section':      str(row.get('report_section', '')),
        'project_count':       int(_safe_float(row.get('supporting_project_count', 0))),
        'pct_of_total':        round(_safe_float(row.get('supporting_project_count', 0)) / max(len(df), 1), 4),
        'verified_topic_count':int(_safe_float(row.get('verified_topic_count', 0))),
        'mean_topic_share':    round(_safe_float(row.get('mean_topic_share_all_verified_topics', 0)), 3),
        'verification_ratio':  round(_safe_float(row.get('verification_ratio', 0)), 3),
        'looker_url':          str(row.get('looker_url', '')),
        'sep1_val':            seps['sep1_val'],
        'sep1_lbl':            seps['sep1_lbl'],
        'sep2_val':            seps['sep2_val'],
        'sep2_lbl':            seps['sep2_lbl'],
        'tvd': {
            'metro': round(_safe_float(tvd_r.get('metro', 0)), 4),
            'grade': round(_safe_float(tvd_r.get('grade', 0)), 4),
            'efs':   round(_safe_float(tvd_r.get('efs',   0)), 4),
            'race':  round(_safe_float(tvd_r.get('race',  0)), 4),
            'mean':  round(_safe_float(tvd_r.get('mean_tvd', 0)), 4),
        },
        'charts': charts_out,
    })

# sort: cross-category first, then by project count
def _sort_key(x):
    rs = x.get('report_section', '').lower()
    return (0 if x['is_cross'] else 1, 0 if 'main' in rs else 1, -x['project_count'])
insights_out.sort(key=_sort_key)

# ── account metadata ──────────────────────────────────────────────────────────
account_meta_path = ROOT / 'account_meta.json'
if account_meta_path.exists():
    account_meta = json.loads(account_meta_path.read_text())
else:
    account_meta = {'name': '', 'logo': '', 'date': RUN_DATE}


# ── payload ───────────────────────────────────────────────────────────────────
payload = {
    'meta': {
        'run_id':           RUN_ID,
        'run_date':         RUN_DATE,
        'project_count':    len(df),
        'insight_count':    len(curated_df),
        'groupby_field':    GROUPBY_FIELD,
        'filter_headline':  filter_headline,
        'filter_detail':    filter_detail,
    },
    'account':     account_meta,
    'full_corpus': fc_charts_local,
    'insights':    insights_out,
}


# ── build ─────────────────────────────────────────────────────────────────────
template_path = ROOT / 'report_template.html'
chartjs_path  = ROOT / 'chart.umd.min.js'

if not template_path.exists():
    raise FileNotFoundError(f"report_template.html not found at {template_path}")
if not chartjs_path.exists():
    raise FileNotFoundError(
        f"chart.umd.min.js not found at {chartjs_path}\n"
        "Run: urllib.request to download from cdn.jsdelivr.net (see prior cell)"
    )

template = template_path.read_text(encoding='utf-8')
chartjs  = chartjs_path.read_text(encoding='utf-8')

html_out = (template
    .replace('__CHARTJS__',    chartjs)
    .replace('__FAVICON__',    favicon_uri)
    .replace('__REPORT_DATA__', json.dumps(payload, ensure_ascii=False, separators=(',', ':')))
)

html_path = OUT('reports', 'trend_tracker_report.html')
html_path.parent.mkdir(parents=True, exist_ok=True)
html_path.write_text(html_out, encoding='utf-8')

size_kb   = len(html_out.encode('utf-8')) / 1024
body_ct   = sum(1 for i in insights_out if i['what_we_see'])
sep_ct    = sum(1 for i in insights_out if i['sep1_val'] is not None)
cross_ct  = sum(1 for i in insights_out if i['is_cross'])

print(f"HTML report written → {html_path}")
print(f"File size: {size_kb:.0f} KB  ·  Insights: {len(insights_out)}  ·  Projects: {len(df):,}")
print(f"Body text: {body_ct}/{len(insights_out)}  ·  Separators: {sep_ct}/{len(insights_out)}  ·  Cross-category: {cross_ct}")
print(f"Filter headline: {filter_headline[:100]}...")


HTML report written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/project_category/2026-04-10/20260410_111800_project_category_77e319db/reports/trend_tracker_report.html
File size: 394 KB  ·  Insights: 44  ·  Projects: 420,625
Body text: 44/44  ·  Separators: 44/44  ·  Cross-category: 8
Filter headline: Insights were found by deep diving on project category, filtered on funded_date from 2023-07-01 to p...
